# Python Data Science Handbook — Ch. 2–3 Reading Notes

**Jake VanderPlas, *Python Data Science Handbook*** — Chapter 2 (Introduction to NumPy) & Chapter 3 (Data Manipulation with Pandas)

Week 1, Month 1 — Personal Claude ML/AI class. Source: https://jakevdp.github.io/PythonDataScienceHandbook/

*Grad-student reading notes (Claude-generated run-through) plus captured session Q&A. Faithful to the source; anything beyond it is flagged* `(added context)`.

Code cells import `numpy as np` / `pandas as pd` where needed and are meant to run top-to-bottom within each section.

---

## Contents

**Ch. 2 — NumPy:** 01 Data types · 02 Array basics · 03 Ufuncs · 04 Aggregations · 05 Broadcasting · 06 Boolean masks · 07 Fancy indexing · 08 Sorting · 09 Structured arrays

**Ch. 3 — Pandas:** 10 Pandas objects · 11 Indexing & selection · 12 Operating on data · 13 Missing data · 14 Hierarchical indexing · 15 Concat & append · 16 Merge & join · 17 Aggregation & grouping · 18 Pivot tables · 19 String ops · 20 Time series · 21 eval/query

# Notes 01 — Understanding Data Types in Python (Vanderplas Ch. 2.1)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.01-understanding-data-types.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Effective data-heavy computing requires understanding *how* Python stores and manipulates data. Python's flexibility (dynamic typing) comes at a cost: every value is a full object with bookkeeping overhead, and standard containers (`list`) store pointers to scattered objects. NumPy fixes this by providing **fixed-type, contiguous arrays** that trade flexibility for the efficiency needed for numerical work. This section explains the trade-off and shows the many ways to build NumPy arrays.

## Key ideas / idioms
- **Dynamic typing:** a Python variable can hold any type and change type freely (`x = 4`, then `x = "four"` is fine). In a statically-typed language like C you must declare the type up front and it cannot change.
- **A Python integer is not just an integer.** It is a pointer to a C structure (`struct _longobject`) holding:
  - `ob_refcnt` — reference count (for memory management)
  - `ob_type` — the variable's type
  - `ob_size` — size of the following data members
  - `ob_digit` — the actual integer value
  This makes Python ints much larger than a raw C int (which is just bytes in memory).
- **A Python list is not just a list.** Because each element is a full Python object, a list stores a pointer to a block of pointers, each pointing to its own object. This allows **heterogeneous** lists but adds memory + indirection overhead.
- **Fixed-type arrays** store a single pointer to one **contiguous block** of data of one type. Much more efficient, but homogeneous (all same type).
- Python ships a built-in `array` module for compact same-type arrays, but **NumPy's `ndarray`** adds efficient *operations* on that data — that's why we use NumPy.
- When making an array from a list, **types are upcast** to a common type if possible (e.g., an int mixed with floats becomes all floats).
- You can force a type with the `dtype` keyword.
- NumPy arrays can be **multidimensional** (nested lists become 2D, etc.).
- Types can be specified as strings (`dtype='int16'`) or NumPy objects (`dtype=np.int16`).
- **No math required for this section**; conceptually, list vs array overhead scales as $$O(n)$$ extra pointers/objects for a list of length $n$, vs a single contiguous buffer for an array. *(added context: the big-O framing is mine, not in the text.)*

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- Dynamic typing (pure Python) ---
x = 4
x = "four"   # legal in Python; would be a type error in C

# --- Built-in array module (same-type, compact) ---
import array
L = list(range(10))
A = array.array('i', L)   # 'i' => integer type code
print(A)

# --- Creating NumPy arrays from Python lists ---
print(np.array([1, 4, 2, 5, 3]))

# Integers get upcast to float when mixed with floats:
print(np.array([3.14, 4, 2, 3]))

# Force a data type explicitly:
print(np.array([1, 2, 3, 4], dtype='float32'))

# Nested lists -> multidimensional array:
print(np.array([range(i, i + 3) for i in [2, 4, 6]]))

# --- Creating arrays from scratch ---
print(np.zeros(10, dtype=int))                 # length-10 array of zeros
print(np.ones((3, 5), dtype=float))            # 3x5 array of ones
print(np.full((3, 5), 3.14))                   # 3x5 array filled with 3.14
print(np.arange(0, 20, 2))                     # like range(): start, stop, step
print(np.linspace(0, 1, 5))                    # 5 values evenly spaced in [0, 1]
print(np.random.random((3, 3)))                # uniform random in [0, 1)
print(np.random.normal(0, 1, (3, 3)))          # normal(mean=0, std=1)
print(np.random.randint(0, 10, (3, 3)))        # random ints in [0, 10)
print(np.eye(3))                               # 3x3 identity matrix
print(np.empty(3))                             # uninitialized (whatever is in memory)

# --- Specifying NumPy data types ---
print(np.zeros(5, dtype='int16'))              # via string
print(np.zeros(5, dtype=np.int16))             # via NumPy object

## Why this matters / intuition
- For data science you operate on *millions* of values; the per-object overhead of Python lists makes element-wise math slow and memory-hungry.
- A contiguous, single-type buffer is what lets NumPy push work down to fast compiled (C/Fortran) loops — this is the foundation everything later in the book (vectorization, broadcasting, pandas) is built on.
- Knowing `dtype` exists matters for **memory** (e.g., `int8` vs `int64`) and for avoiding silent precision surprises.

## Gotchas
- **Upcasting is silent:** `np.array([3.14, 4, 2, 3])` quietly becomes all floats. If you expected ints, set `dtype` explicitly.
- **NumPy arrays are homogeneous.** Unlike a Python list, you can't freely mix types in one array.
- **`np.empty` is NOT zeros.** It returns whatever happens to already be in that memory — never assume initial values.
- **`np.arange` stop is exclusive**, and with float steps it can hit floating-point edge cases; prefer `np.linspace` when you need an exact number of evenly spaced points.
- A NumPy array indexed/sliced will coerce assigned values to the array's `dtype` (e.g., writing a float into an int array truncates). *(added context: implied by fixed-type storage; the explicit truncation example appears in the next sections.)*

## Suggested figure (optional)
Two side-by-side memory diagrams:
1. **Python list:** a contiguous block of pointers, each arrow leading off to a separately-allocated Python integer object (each object box showing `ob_refcnt / ob_type / ob_size / ob_digit`). Emphasizes scattered memory + double indirection.
2. **NumPy array:** a single header (dtype, shape, strides) pointing to one contiguous block of raw values packed back-to-back. Emphasizes one allocation, no per-element object overhead.
This is the classic "C int vs Python int / list vs array" layout that makes the efficiency argument visual.

# Notes 02 — The Basics of NumPy Arrays (Vanderplas Ch. 2.2)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
This section covers manipulating NumPy arrays once you have them. It groups operations into five basic categories: inspecting **array attributes**, **indexing** single elements, **slicing** subarrays, **reshaping**, and **joining/splitting** arrays. The recurring theme is that NumPy arrays are fixed-type, fixed-size, contiguous data, which makes them fast but introduces behaviors (silent truncation, views-not-copies) that differ from Python lists.

## Key ideas / idioms
- **Attributes describe structure:** every array exposes `ndim` (number of dimensions), `shape` (tuple of per-axis sizes), `size` (total element count), `dtype` (element type), `itemsize` (bytes per element), and `nbytes` (total bytes). Note $$\text{nbytes} = \text{size} \times \text{itemsize}.$$
- **Indexing** uses brackets; multidimensional access uses a comma-separated tuple `x2[row, col]`. Negative indices count from the end.
- **Fixed type:** assigning a float into an int array silently **truncates** it — no automatic upcasting like a Python list.
- **Slicing** uses `x[start:stop:step]` (defaults: `start=0`, `stop=size`, `step=1`). A negative `step` reverses; a common idiom for full reversal is `x[::-1]`.
- **Slices are VIEWS, not copies.** Modifying a slice modifies the original array. Use `.copy()` to break the link. (This is what makes working with large datasets cheap — no implicit copying.)
- **Reshaping** with `reshape` requires the new shape's size to match; `np.newaxis` (or `reshape`) converts a 1D array into a row or column vector.
- **Concatenation:** `np.concatenate` joins along an existing axis (default `axis=0`); `np.vstack`/`np.hstack`/`np.dstack` stack along axes 0/1/2 and handle mixed dimensions cleanly.
- **Splitting** is the inverse: `np.split` (with `np.hsplit`/`np.vsplit`/`np.dsplit`); $N$ split-points yield $N+1$ subarrays.

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- Array attributes ---
np.random.seed(0)
x1 = np.random.randint(10, size=6)         # 1D
x2 = np.random.randint(10, size=(3, 4))    # 2D
x3 = np.random.randint(10, size=(3, 4, 5)) # 3D

print("x3 ndim: ", x3.ndim)        # 3
print("x3 shape:", x3.shape)       # (3, 4, 5)
print("x3 size: ", x3.size)        # 60
print("dtype:   ", x3.dtype)       # int64 (platform-dependent)
print("itemsize:", x3.itemsize, "bytes")
print("nbytes:  ", x3.nbytes, "bytes")  # == size * itemsize

In [ ]:
import numpy as np

# --- Indexing ---
x1 = np.array([5, 0, 3, 3, 7, 9])
print(x1[0], x1[4], x1[-1])   # 5 7 9

x2 = np.array([[3, 5, 2, 4],
               [7, 6, 8, 8],
               [1, 6, 7, 7]])
print(x2[0, 0], x2[2, -1])    # 3 7
x2[0, 0] = 12                 # modify in place

# Fixed-type quirk: float is truncated when stored in an int array
x1[0] = 3.14159
print(x1)                     # [3 0 3 3 7 9]

In [ ]:
import numpy as np

# --- Slicing (1D) ---
x = np.arange(10)
print(x[:5])     # [0 1 2 3 4]
print(x[5:])     # [5 6 7 8 9]
print(x[4:7])    # [4 5 6]
print(x[::2])    # [0 2 4 6 8]
print(x[1::2])   # [1 3 5 7 9]
print(x[::-1])   # [9 8 7 6 5 4 3 2 1 0]
print(x[5::-2])  # [5 3 1]

# --- Slicing (multi-dim) ---
x2 = np.array([[12, 5, 2, 4],
               [7, 6, 8, 8],
               [1, 6, 7, 7]])
print(x2[:2, :3])      # first 2 rows, first 3 cols
print(x2[:3, ::2])     # all rows, every other col
print(x2[::-1, ::-1])  # reverse both axes
print(x2[:, 0])        # first column -> [12 7 1]
print(x2[0])           # first row (same as x2[0, :]) -> [12 5 2 4]

In [ ]:
import numpy as np

# --- Views vs copies ---
x2 = np.array([[12, 5, 2, 4],
               [7, 6, 8, 8],
               [1, 6, 7, 7]])
x2_sub = x2[:2, :2]
x2_sub[0, 0] = 99
print(x2[0, 0])        # 99  <- original changed (slice was a view)

x2_sub_copy = x2[:2, :2].copy()
x2_sub_copy[0, 0] = 42
print(x2[0, 0])        # still 99 <- copy is independent

In [ ]:
import numpy as np

# --- Reshaping ---
grid = np.arange(1, 10).reshape((3, 3))
print(grid)

x = np.array([1, 2, 3])
print(x[np.newaxis, :])  # row vector, shape (1, 3)
print(x[:, np.newaxis])  # column vector, shape (3, 1)

In [ ]:
import numpy as np

# --- Concatenation ---
x = np.array([1, 2, 3])
y = np.array([3, 2, 1])
print(np.concatenate([x, y]))            # [1 2 3 3 2 1]
print(np.concatenate([x, y, [99,99,99]]))

grid = np.array([[1, 2, 3],
                 [4, 5, 6]])
print(np.concatenate([grid, grid]))          # stack rows (axis=0)
print(np.concatenate([grid, grid], axis=1))  # stack cols (axis=1)

# Mixed dimensions: use vstack / hstack
row = np.array([1, 2, 3])
g = np.array([[9, 8, 7],
              [6, 5, 4]])
print(np.vstack([row, g]))
col = np.array([[99], [99]])
print(np.hstack([g, col]))

In [ ]:
import numpy as np

# --- Splitting ---
x = np.array([1, 2, 3, 99, 99, 3, 2, 1])
a, b, c = np.split(x, [3, 5])   # 2 split-points -> 3 subarrays
print(a, b, c)                  # [1 2 3] [99 99] [3 2 1]

grid = np.arange(16).reshape((4, 4))
upper, lower = np.vsplit(grid, [2])
left, right = np.hsplit(grid, [2])
print(upper); print(lower)
print(left);  print(right)

## Why this matters / intuition
Almost all real data work — loading datasets, selecting features/rows, building batches, assembling matrices — reduces to these five operations. Understanding that **slices are views** is the single most important takeaway: it makes NumPy efficient (selecting a million-row chunk costs nothing), but it also means careless slicing can silently mutate your source data. Knowing when NumPy gives you a window versus a fresh array is the difference between fast, correct code and subtle data-corruption bugs.

## Gotchas
- **Silent truncation:** putting a float in an int array drops the fractional part with no warning. Choose `dtype` deliberately.
- **Views, not copies:** `x[:2, :2]` shares memory with `x`. Mutating it mutates the original — use `.copy()` when you need independence. (Note: *fancy* indexing, covered later, returns copies, not views — a separate behavior.) (added context)
- **Split-point counting:** `np.split(x, [3, 5])` returns *3* pieces, not 2 — $N$ indices give $N+1$ slices.
- **`reshape` size must match:** the product of the new shape must equal the original `size`, or it errors. (`-1` can be used to infer one dimension automatically.) (added context)
- **`dtype`/`nbytes` are platform-dependent:** default integer width may print as `int64` or `int32` depending on OS/build. (added context)

## Suggested figure (optional)
A side-by-side diagram of a small 2D array and a highlighted slice, with two arrows: one labeled "view (shares memory — edits propagate back)" pointing from slice to original, and one labeled ".copy() (independent block — edits stay local)" showing a separate detached memory region. This visually anchors the views-vs-copies distinction.

# Notes 03 — Computation on Arrays: Universal Functions (Vanderplas Ch. 2.3)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.03-computation-on-arrays-ufuncs.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

CPython is slow on element-by-element loops because its dynamic typing forces type-checking and function dispatch on every iteration — the cost is the per-element overhead, not the arithmetic itself. NumPy fixes this with **vectorized operations** implemented as **universal functions (ufuncs)**: a single statement (e.g. `1.0 / values`) pushes the loop down into a compiled layer, executing the same computation hundreds of times faster.

A naive Python-loop reciprocal over 1 million values takes ~2.91 s; the vectorized `1.0 / values` takes ~4.6 ms — over 600x faster. The speedup grows with array size.

ufuncs come in two flavors: **unary** (one input, e.g. `np.negative`) and **binary** (two inputs, e.g. `np.add`). They work on scalar-and-array, array-and-array, and multi-dimensional arrays alike.

## Key ideas / idioms

- Vectorized expression replaces an explicit loop; the loop runs in compiled code.
- All standard Python operators are wrapped as ufuncs and obey normal precedence / can be chained.
- Operators ↔ ufuncs:

| Operator | ufunc | Operation |
|----------|-------|-----------|
| `+` | `np.add` | addition |
| `-` | `np.subtract` | subtraction |
| `-` (unary) | `np.negative` | negation |
| `*` | `np.multiply` | multiplication |
| `/` | `np.divide` | division |
| `//` | `np.floor_divide` | floor division |
| `**` | `np.power` | exponentiation |
| `%` | `np.mod` | modulus |

- Math families: `np.abs`/`np.absolute`; trig (`np.sin/cos/tan`, `np.arcsin/arccos/arctan`); exponents (`np.exp`, `np.exp2`, `np.power`); logs (`np.log`, `np.log2`, `np.log10`); precise small-value variants `np.expm1`, `np.log1p`.
- For complex input, `np.abs` returns the magnitude $|a+bi| = \sqrt{a^2+b^2}$.
- Advanced ufunc methods: `out=` (write in place), `.reduce()` (collapse to one value), `.accumulate()` (keep intermediates), `.outer()` (all input pairs).
- `expm1`/`log1p` give better precision near zero:
$$\texttt{expm1}(x) = e^x - 1, \qquad \texttt{log1p}(x) = \ln(1+x)$$

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- Arithmetic operators and their ufunc equivalents ---
x = np.arange(4)            # [0 1 2 3]
print(x + 5)                # [5 6 7 8]
print(x * 2)                # [0 2 4 6]
print(x ** 2)               # [0 1 4 9]
print(-(0.5 * x + 1) ** 2)  # [-1. -2.25 -4. -6.25]
print(np.add(x, 5))         # same as x + 5
print(np.multiply(x, 2))    # same as x * 2

In [ ]:
import numpy as np

# --- Absolute value (real and complex) ---
x = np.array([-2, -1, 0, 1, 2])
print(abs(x))          # [2 1 0 1 2]
print(np.abs(x))       # [2 1 0 1 2]  (np.absolute is the same)

z = np.array([3 - 4j, 4 - 3j, 2 + 0j, 0 + 1j])
print(np.abs(z))       # [5. 5. 2. 1.]  (magnitudes)

In [ ]:
import numpy as np

# --- Trigonometric functions ---
theta = np.linspace(0, np.pi, 3)
print(np.sin(theta))   # [0. 1. ~0]
print(np.cos(theta))   # [1. ~0 -1.]
print(np.tan(theta))   # [0. ~large ~0]

xs = [-1, 0, 1]
print(np.arcsin(xs))   # [-1.5708  0.  1.5708]
print(np.arccos(xs))   # [3.1416  1.5708  0.]
print(np.arctan(xs))   # [-0.7854  0.  0.7854]

In [ ]:
import numpy as np

# --- Exponents and logarithms ---
x = [1, 2, 3]
print(np.exp(x))       # [2.718  7.389  20.086]
print(np.exp2(x))      # [2. 4. 8.]
print(np.power(3, x))  # [3 9 27]

x = [1, 2, 4, 10]
print(np.log(x))       # natural log
print(np.log2(x))      # [0. 1. 2. 3.3219]
print(np.log10(x))     # [0. 0.301 0.602 1.]

# Precise versions for tiny inputs
x = [0, 0.001, 0.01, 0.1]
print(np.expm1(x))     # e^x - 1
print(np.log1p(x))     # ln(1 + x)

In [ ]:
import numpy as np
from scipy import special

# --- Specialized ufuncs via scipy.special ---
x = [1, 5, 10]
print(special.gamma(x))     # [1. 24. 362880.]
print(special.gammaln(x))   # log of gamma
print(special.beta(x, 2))

x = np.array([0, 0.3, 0.7, 1.0])
print(special.erf(x))       # error function
print(special.erfc(x))      # complement
print(special.erfinv(x))    # inverse (last -> inf)

In [ ]:
import numpy as np

# --- Advanced: specifying output with out= ---
x = np.arange(5)
y = np.empty(5)
np.multiply(x, 10, out=y)
print(y)               # [0. 10. 20. 30. 40.]

y = np.zeros(10)
np.power(2, x, out=y[::2])   # write into a strided view
print(y)               # [1. 0. 2. 0. 4. 0. 8. 0. 16. 0.]

In [ ]:
import numpy as np

# --- Advanced: reduce, accumulate, outer ---
x = np.arange(1, 6)            # [1 2 3 4 5]
print(np.add.reduce(x))        # 15
print(np.multiply.reduce(x))   # 120
print(np.add.accumulate(x))    # [1 3 6 10 15]
print(np.multiply.accumulate(x))  # [1 2 6 24 120]

print(np.multiply.outer(x, x)) # 5x5 multiplication table

## Why this matters / intuition

The performance bottleneck in pure-Python numerics is overhead, not math: each loop iteration pays for dynamic type-checking and function dispatch. ufuncs move the loop into compiled C, so the per-element overhead vanishes and the CPU runs tight, cache-friendly code. The practical rule from the chapter: *when you see a loop over a NumPy array, ask whether it can be rewritten as a vectorized ufuncs expression* — it is nearly always faster, increasingly so as arrays grow.

The `out=` argument matters for large arrays because it avoids allocating a temporary result array, saving memory and a copy. *(added context)* This is the same in-place-write idea behind augmented assignment like `x += 1`.

## Gotchas

- Floating-point round-off: `np.sin(np.pi)` returns ~1.22e-16, not exactly 0 — values near machine precision are effectively zero.
- Use `np.expm1`/`np.log1p` instead of `np.exp`/`np.log` for very small inputs; the plain versions lose precision near zero.
- `out=` requires the target array to already exist with the right shape/dtype; it does not allocate for you. It also works with views (e.g. `y[::2]`), writing into strided memory.
- `scipy.special` is a separate import (`from scipy import special`), not part of NumPy.
- `.reduce()` and `.accumulate()` are ufunc *methods*; for common cases prefer the dedicated `np.sum`, `np.prod`, `np.cumsum`, `np.cumprod` (covered next section).

## Suggested figure (optional)

A two-bar (log-scale) timing chart contrasting the Python-loop reciprocal (~2.91 s) against the vectorized `1.0 / values` (~4.6 ms) over 1,000,000 elements, with the ~600x gap annotated — visually driving home that vectorization wins, and by how much.

---

## 💬 Q&A (captured during session)

### Q: What are ufuncs?

**Ufuncs (universal functions)** are NumPy's vectorized, element-wise operations — functions that apply the *same* operation to every element of an array (or pair of arrays) in one go, via fast precompiled C loops instead of a Python `for` loop.

**The problem they solve.** Python loops are slow because every iteration re-does type-checking and dispatch (the dynamic-typing overhead from Notes 01). For an array, that per-element overhead dominates:

In [ ]:
import numpy as np

big = np.random.randint(1, 100, size=1_000_000)

# Slow: Python-level loop, type-checked every element
def reciprocals_loop(arr):
    out = np.empty(len(arr))
    for i in range(len(arr)):
        out[i] = 1.0 / arr[i]
    return out

# Fast: one ufunc call, C loop under the hood
fast = 1.0 / big          # this is the ufunc np.divide

Same result, but the ufunc version is typically tens to hundreds of times faster — the loop happens in C, once.

**Two flavors:**
- **Unary** — operate on one array: `np.abs`, `np.exp`, `np.log`, `np.sin`, `-x`
- **Binary** — operate on two: `np.add`, `np.multiply`, `np.power`, `x > y`

**Key idea: operators *are* ufuncs.** The arithmetic you already write is just sugar for ufuncs:

In [ ]:
x = np.arange(4)        # [0 1 2 3]
x + 5                   # np.add(x, 5)        -> [5 6 7 8]
x ** 2                  # np.power(x, 2)      -> [0 1 4 9]
x % 2                   # np.mod(x, 2)        -> [0 1 0 1]

**Handy advanced features:**

In [ ]:
x = np.arange(1, 6)               # [1 2 3 4 5]

out = np.empty(5)
np.multiply(x, 10, out=out)       # write straight into out, no temp array

np.add.reduce(x)                  # 15  (sum all -> like np.sum)
np.add.accumulate(x)              # [ 1  3  6 10 15]  (running sum)
np.multiply.outer(x, x)           # full multiplication table (5x5)

**Mental model:** whenever you're tempted to loop over a NumPy array, stop — there's almost always a ufunc (or combination) that does it element-wise in C. *"Vectorize, don't iterate"* is the core NumPy idiom, and ufuncs are the machinery behind it. They also obey **broadcasting** (Notes 05).

### Q: What are wrappers?

A **wrapper** is a thin piece of code that goes *around* another piece of code to provide a more convenient or different interface to it — without changing what the underlying thing actually does. It "wraps" the real work and hands off to it.

This came up above: when `x + 5` is "sugar for `np.add(x, 5)`", the `+` operator is acting as a **wrapper** around the `np.add` ufunc.

**The idea — a wrapper:**
1. Takes a call in a simple/familiar form,
2. maybe does a little setup or translation,
3. calls the real ("wrapped") function that does the heavy lifting,
4. hands back the result.

You get an easier interface; the actual computation is unchanged.

**NumPy example.** NumPy makes the `+`, `-`, `**`, `%`, `>` operators *wrap* the corresponding ufuncs so you can write natural math instead of verbose calls:

In [ ]:
import numpy as np

x = np.arange(4)

x + 5            # convenient wrapper...
np.add(x, 5)     # ...around this actual ufunc — identical result

x ** 2           # wrapper around np.power(x, 2)
-x               # wrapper around np.negative(x)

*(added context — general programming concept, not from the book)* A wrapper in plain Python, the pattern behind **decorators**:

In [ ]:
import time

def slow_add(a, b):
    time.sleep(0.1)
    return a + b

# A "timing wrapper" around slow_add
def timed(func):
    def wrapper(*args, **kwargs):          # same call signature
        start = time.perf_counter()
        result = func(*args, **kwargs)     # call the wrapped function
        print(f"took {time.perf_counter() - start:.3f}s")
        return result                      # hand back its result
    return wrapper

fast_to_call = timed(slow_add)
fast_to_call(2, 3)     # prints timing, still returns 5

Here `wrapper` adds timing *around* `slow_add` but doesn't change the addition itself.

**Why it matters.** Wrappers are everywhere in data-science code:
- **Operators wrapping ufuncs** → readable math.
- **Pandas methods wrapping NumPy** — e.g. `df.sum()` wraps `np.sum` but adds index/label handling (Notes 12).
- **Convenience constructors** — `pd.read_csv` wraps a lot of parsing machinery behind one call.

The mental model: *a wrapper changes how you call something, not what it ultimately does.*

# Notes 04 — Aggregations: Min, Max, and Everything In Between (Vanderplas Ch. 2.4)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.04-computation-on-arrays-aggregates.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Aggregations reduce an array down to a single summary value (or a row/column of summary values): sum, min, max, mean, std, etc. NumPy ships compiled aggregation functions that run much faster than Python's built-ins and that are dimension-aware (they understand the `axis` argument). These summaries are the first thing you compute when exploring a dataset.

## Key ideas / idioms
- **Use NumPy's versions, not Python built-ins.** `np.sum(L)` runs ~200x faster than the built-in `sum(L)` on a large array because it executes in compiled code. They also differ in optional arguments and dimensional awareness, so they are *not* interchangeable.
- **Two call styles.** Functional form `np.min(arr)` / `np.max(arr)` and the shorter method form `arr.min()` / `arr.max()`. Same result.
- **`axis` collapses a dimension.** For 2D arrays, `axis` names *the dimension that gets collapsed*, not the one that remains. So `axis=0` collapses rows → one value per column; `axis=1` collapses columns → one value per row. With no `axis`, the whole array is reduced to a scalar.
- **NaN-safe variants.** Most functions have an `np.nan*` twin that ignores missing values (`NaN`) instead of letting them poison the result.
- For a column of values $x_1, \dots, x_n$, the mean and (population) standard deviation are:
$$ \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad \sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2} $$
*(added context: NumPy's `np.std` uses the population formula — divisor $n$ — by default; pass `ddof=1` for the sample standard deviation with divisor $n-1$.)*

### Aggregation functions table
| Function | NaN-safe version | Purpose |
|----------|------------------|---------|
| `np.sum` | `np.nansum` | Sum of elements |
| `np.prod` | `np.nanprod` | Product of elements |
| `np.mean` | `np.nanmean` | Mean of elements |
| `np.std` | `np.nanstd` | Standard deviation |
| `np.var` | `np.nanvar` | Variance |
| `np.min` | `np.nanmin` | Minimum value |
| `np.max` | `np.nanmax` | Maximum value |
| `np.argmin` | `np.nanargmin` | Index of minimum |
| `np.argmax` | `np.nanargmax` | Index of maximum |
| `np.median` | `np.nanmedian` | Median value |
| `np.percentile` | `np.nanpercentile` | Rank-based quantile |
| `np.any` | N/A | Whether any element is true |
| `np.all` | N/A | Whether all elements are true |

## Worked code examples (runnable)

In [ ]:
import numpy as np

# NumPy aggregation vs Python built-in (same value, very different speed)
L = np.random.random(100)
print(sum(L))        # Python built-in
print(np.sum(L))     # NumPy version (compiled, ~200x faster on large arrays)

# Min and max, functional form and method form
big_array = np.random.rand(1_000_000)
print(np.min(big_array), np.max(big_array))
print(big_array.min(), big_array.max())   # shorthand methods

In [ ]:
import numpy as np

# Multidimensional aggregates: axis names the dimension that is COLLAPSED
M = np.random.random((3, 4))
print(M)
print("whole array sum :", M.sum())        # single scalar
print("min of each col :", M.min(axis=0))  # collapse rows -> 4 values
print("max of each row :", M.max(axis=1))  # collapse cols -> 3 values

In [ ]:
import numpy as np

# Summary-statistics example (mirrors the presidents'-heights walkthrough).
# Self-contained: we synthesize heights instead of reading the CSV.
heights = np.array([189, 170, 189, 163, 183, 171, 185, 168, 173, 183])

print("Mean height       :", heights.mean())
print("Standard deviation:", heights.std())
print("Minimum height    :", heights.min())
print("Maximum height    :", heights.max())
print("25th percentile   :", np.percentile(heights, 25))
print("Median            :", np.median(heights))
print("75th percentile   :", np.percentile(heights, 75))

*(added context: in the book this data comes from `pd.read_csv('data/president_heights.csv')` and `np.array(data['height(cm)'])`, giving mean ≈ 179.74 cm, std ≈ 6.93, min 163, max 193. The array above is a stand-in so the snippet runs without the file.)*

## Why this matters / intuition
Before modeling anything, you describe it. A handful of aggregates — mean, spread, extremes, quartiles — turns a raw column of numbers into something you can reason about and sanity-check. Doing this in compiled NumPy (and along chosen axes) makes it fast enough to apply to large datasets interactively, which is exactly the loop exploratory data analysis depends on.

## Gotchas
- `sum` vs `np.sum` are *different functions*. Built-in `sum` is slow and does not understand `axis`; prefer `np.sum`.
- Mixing them up on multidimensional arrays gives confusing results because only `np.sum` interprets `axis`.
- The `axis` argument is easy to invert: `axis=0` does **not** mean "operate over rows to get a per-row answer" — it *collapses* the row dimension, yielding a per-column result.
- A single `NaN` makes ordinary aggregations return `NaN`. Use the `np.nan*` variants when missing data is possible.
- *(added context)* `np.std` defaults to population std (`ddof=0`); for an unbiased sample estimate use `ddof=1`.

## Suggested figure (optional)
A 3x4 grid of cells with arrows: a downward arrow over the columns labeled `axis=0 -> one value per column`, and a rightward arrow over the rows labeled `axis=1 -> one value per row`, with a separate note that "no axis" reduces the whole grid to a single number. This visually reinforces that `axis` marks the dimension being collapsed.

---

## 💬 Q&A (captured during session)

### Q: What does "aggregate" mean?

**Aggregate** means to **combine many values into a single summary value**. An aggregation takes a whole collection of numbers and boils it down to *one* number (or a small set of numbers) that summarizes them. The word itself = "to gather into a whole": you *gather* many data points and collapse them into a representative figure.

**Everyday examples** — given `[5, 2, 8, 1, 9]`:
- **sum** → `25` (all added) · **mean** → `5.0` · **min** → `1`, **max** → `9` · **count** → `5`

Each is an *aggregate*: many in, one out.

In [ ]:
import numpy as np

x = np.array([5, 2, 8, 1, 9])
x.sum()      # 25   -> one number
x.mean()     # 5.0
x.min()      # 1
x.max()      # 9
x.std()      # spread around the mean

For 2-D arrays you aggregate along an **axis** — collapsing just one dimension instead of everything:

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])

M.sum()           # 21   (everything -> one number)
M.sum(axis=0)     # [5 7 9]   (collapse rows -> one value per column)
M.sum(axis=1)     # [6 15]    (collapse columns -> one value per row)

Mental model for `axis`: it names the dimension that **disappears**. `axis=0` collapses the rows, leaving column totals.

**Why it matters:** aggregation is usually the *first* thing you do to understand a dataset — you can't eyeball a million rows, but you can look at their mean/min/max/std. In Pandas (**Notes 17**) aggregation becomes **split-apply-combine**: split into groups, aggregate each group separately (e.g. average sales *per region*), then combine the per-group summaries into one table.

# Notes 05 — Computation on Arrays: Broadcasting (Vanderplas Ch. 2.5)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Broadcasting is simply a set of rules for applying binary ufuncs (addition, subtraction, multiplication, etc.) on arrays of **different sizes**. For equal-sized arrays, ufuncs operate element-by-element. Broadcasting extends this so that smaller arrays are conceptually "stretched" to match the shape of the larger one — but the stretching is virtual: NumPy does **not** actually duplicate data in memory, which keeps broadcasting efficient.

## Key ideas / idioms

- Same-size arrays operate element-wise: `a + b` matches positions directly.
- A scalar broadcasts against an array as if the scalar were stretched to the array's shape (e.g., `a + 5`).
- Broadcasting also stretches **both** operands when needed (e.g., a column vector plus a row vector → a 2D grid).
- Use `np.newaxis` (or `reshape`) to add a length-1 axis when you need to control *which* dimension stretches.

**The three rules of broadcasting:**

$$
\textbf{Rule 1:}\quad \text{If two arrays differ in number of dimensions, the shape of the array with fewer dimensions is padded with ones on its \emph{leading (left) side}.}
$$

$$
\textbf{Rule 2:}\quad \text{If the shapes disagree in any dimension, the array with shape equal to 1 in that dimension is stretched to match the other shape.}
$$

$$
\textbf{Rule 3:}\quad \text{If in any dimension the sizes disagree and neither is equal to 1, an error is raised.}
$$

## Worked code examples (runnable)

In [ ]:
import numpy as np

# Element-wise on same-size arrays
a = np.array([0, 1, 2])
b = np.array([5, 5, 5])
print(a + b)        # [5 6 7]

# Scalar broadcast (scalar "stretched" to shape (3,))
print(a + 5)        # [5 6 7]

In [ ]:
import numpy as np

# Example 1: 2D + 1D
# M.shape = (2, 3), a.shape = (3,)
#   Rule 1: a -> (1, 3)
#   Rule 2: a -> (2, 3)
M = np.ones((2, 3))
a = np.arange(3)
print(M + a)
# [[1. 2. 3.]
#  [1. 2. 3.]]

In [ ]:
import numpy as np

# Example 2: both arrays broadcast
# a.shape = (3, 1), b.shape = (3,)
#   Rule 1: b -> (1, 3)
#   Rule 2: a -> (3, 3), b -> (3, 3)
a = np.arange(3).reshape((3, 1))
b = np.arange(3)
print(a + b)
# [[0 1 2]
#  [1 2 3]
#  [2 3 4]]

In [ ]:
import numpy as np

# Example 3: incompatible shapes -> error
# M.shape = (3, 2), a.shape = (3,)
#   Rule 1: a -> (1, 3)
#   Rule 2: a -> (3, 3)  ... but M is (3, 2): mismatch -> Rule 3 error
M = np.ones((3, 2))
a = np.arange(3)
try:
    M + a
except ValueError as e:
    print("ValueError:", e)
# ValueError: operands could not be broadcast together with shapes (3,2) (3,)

# Fix: add a trailing axis so a becomes (3, 1), which broadcasts to (3, 2)
print(a[:, np.newaxis].shape)   # (3, 1)
print(M + a[:, np.newaxis])
# [[1. 1.]
#  [2. 2.]
#  [3. 3.]]

In [ ]:
import numpy as np

# Broadcasting in practice 1: centering an array
X = np.random.random((10, 3))
Xmean = X.mean(0)            # mean per column, shape (3,)
X_centered = X - Xmean       # (3,) broadcasts across all 10 rows
print(np.allclose(X_centered.mean(0), 0))  # True (means ~ [0, 0, 0])

In [ ]:
import numpy as np

# Broadcasting in practice 2: evaluating a 2D function on a grid
x = np.linspace(0, 5, 50)                 # shape (50,)
y = np.linspace(0, 5, 50)[:, np.newaxis]  # shape (50, 1)
# x -> (1, 50), y -> (50, 1) broadcast to (50, 50)
z = np.sin(x) ** 10 + np.cos(10 + y * x) * np.cos(x)
print(z.shape)   # (50, 50)

# (Visualization, requires matplotlib):
# import matplotlib.pyplot as plt
# plt.imshow(z, origin='lower', extent=[0, 5, 0, 5], cmap='viridis')
# plt.colorbar()

## Why this matters / intuition

Broadcasting lets you write vectorized expressions over arrays of mismatched shapes without manually tiling/looping data. The mental model: align shapes from the **right**, pad the shorter shape with leading ones, then any axis of size 1 gets stretched to match its partner. Because the stretch is virtual (no real copies), you get loop-free, memory-efficient code — invaluable for operations like mean-centering feature matrices and evaluating functions over a coordinate grid.

## Gotchas

- **Padding is on the left only.** Rule 1 always prepends ones. So `(3,)` becomes `(1, 3)`, never `(3, 1)`. That is why `np.ones((3, 2)) + np.arange(3)` fails — `a` becomes `(1, 3)` → `(3, 3)`, which disagrees with `(3, 2)`.
- **Fix mismatches with `np.newaxis`.** To make a 1D array broadcast down columns instead of across rows, give it a trailing axis: `a[:, np.newaxis]` → `(3, 1)`.
- A dimension only stretches if it is exactly **1**; any other disagreement is a hard `ValueError` (Rule 3).
- *(added context)* Broadcasting rules apply to all binary ufuncs uniformly, not just `+` — the same logic governs `*`, `-`, comparisons, etc.

## Suggested figure (optional)

A "broadcasting stretch" diagram: three side-by-side panels, each showing two input grids and a result grid, with light-shaded cells marking duplicated (stretched) values.
- Panel 1 (array + scalar): a 1x3 row of filled boxes plus a single scalar box; the scalar is shown faintly copied into a 1x3 row, producing a 1x3 result.
- Panel 2 (2D + 1D row): a 2x3 grid plus a 1x3 row; the row is shown faintly replicated down to 2 rows, yielding a 2x3 result.
- Panel 3 (column + row): a 3x1 column plus a 1x3 row; the column is faintly tiled rightward and the row faintly tiled downward, both meeting in a full 3x3 result grid.
The faint/ghosted cells should visually emphasize that the stretched values are conceptual copies, not stored in memory.
---

## 💬 Q&A (captured during session)

### Q: Simplify the three broadcasting rules.

**Plain-English version:**

1. **Different number of dimensions?** Pad the smaller array's shape with 1s on the **left** until they match.
2. **A dimension is 1?** Stretch it to match the other array's size in that dimension.
3. **Sizes differ and neither is 1?** Error — they can't broadcast.

**Even shorter:** broadcasting works if, in every dimension, the sizes are either **equal** or **one of them is 1**.

In [ ]:
import numpy as np

# (3,) + scalar      -> pad & stretch    OK
np.arange(3) + 5                 # [5 6 7]

# (3,3) + (3,)       -> pad left to (1,3), stretch to (3,3)   OK
np.ones((3, 3)) + np.arange(3)

# (3,) + (4,)        -> 3 vs 4, neither is 1   ERROR
# np.arange(3) + np.arange(4)

**Mental picture:** line the shapes up **right-to-left**; a `1` (or a missing/padded slot) gets copied to fill the bigger size, but two real, unequal numbers clash.

```
  (3, 3)          (3,)            (3,)
+    (3)   ->   +  (4,)   ->    pad: (1, 3)
---------      --------         stretch: (3, 3)   OK
  (3, 3)        clash
```

# Notes 06 — Comparisons, Masks, and Boolean Logic (Vanderplas Ch. 2.6)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.06-boolean-arrays-and-masks.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Boolean masking is the technique for examining and manipulating array values **based on a criterion**: "Masking comes up when you want to extract, modify, count, or otherwise manipulate values in an array based on some criterion." The workflow is two-step: (1) build a Boolean array with comparison operators (themselves implemented as element-wise ufuncs), then (2) count, test, combine, or index with that Boolean array. VanderPlas motivates it with Seattle 2014 rainfall data — answering "how many rainy days?" or "what's the median precipitation on rainy days?" without slow Python loops.

## Key ideas / idioms

- **Comparisons are ufuncs.** `<`, `>`, `<=`, `>=`, `==`, `!=` all act element-wise and return a Boolean array of the same shape. They work on compound expressions too, e.g. `(2 * x) == (x ** 2)`.

  | Operator | ufunc equivalent |
  |----------|------------------|
  | `==` | `np.equal` |
  | `!=` | `np.not_equal` |
  | `<`  | `np.less` |
  | `<=` | `np.less_equal` |
  | `>`  | `np.greater` |
  | `>=` | `np.greater_equal` |

- **Counting True values:** `np.count_nonzero(mask)` or `np.sum(mask)` (since `False`→0, `True`→1). `np.sum` accepts an `axis`, so `np.sum(x < 6, axis=1)` counts per row.

  $$\text{count} = \sum_i \mathbb{1}[\,\text{condition}_i\,]$$

- **Testing:** `np.any(mask)` ("are *any* True?") and `np.all(mask)` ("are *all* True?"), both axis-aware.

- **Combine conditions with bitwise operators** `&`, `|`, `^`, `~` — **not** `and`/`or`. Always parenthesize: `(inches > 0.5) & (inches < 1)`, because operator precedence would otherwise evaluate the bare expression incorrectly.

  | Operator | ufunc equivalent |
  |----------|------------------|
  | `&` | `np.bitwise_and` |
  | `\|` | `np.bitwise_or` |
  | `^` | `np.bitwise_xor` |
  | `~` | `np.bitwise_not` |

- **Masking = Boolean indexing:** `x[mask]` returns a 1D array of only the elements where `mask` is `True`, e.g. `x[x < 5]`.

- **`and`/`or` vs `&`/`|` (the critical distinction):**
  - `and`/`or` evaluate the truth value of an **entire object**. On a multi-element array this is ambiguous and raises a `ValueError`.
  - `&`/`|` operate **element-wise** (bit-by-bit). "For Boolean NumPy arrays, the latter is nearly always the desired operation."

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- Comparisons return Boolean arrays ---
x = np.array([1, 2, 3, 4, 5])
print(x < 3)              # [ True  True False False False]
print(x >= 3)             # [False False  True  True  True]
print(x == 3)             # [False False  True False False]
print((2 * x) == (x ** 2))  # element-wise comparison of expressions

# --- Counting True entries ---
rng = np.random.RandomState(0)
M = rng.randint(0, 10, (3, 4))
print(M)
print(np.count_nonzero(M < 6))   # total entries < 6
print(np.sum(M < 6))             # same: True->1, False->0
print(np.sum(M < 6, axis=1))     # count per row

# --- Testing conditions ---
print(np.any(M > 8))             # any value > 8?
print(np.all(M < 10))            # all values < 10?
print(np.all(M < 8, axis=1))     # per-row check

# --- Compound conditions (note the parentheses!) ---
between = np.sum((M > 2) & (M < 6))
print("entries in (2, 6):", between)
# De Morgan equivalent using ~ and |
print(np.sum(~((M <= 2) | (M >= 6))))

# --- Boolean arrays as masks ---
print(M[M < 5])                  # 1D array of all values < 5

# --- A rainfall-style analysis on synthetic data ---
inches = rng.rand(365)           # stand-in for daily precipitation
days = np.arange(365)
rainy  = (inches > 0)
summer = (days > 172) & (days < 262)
print("Median precip on rainy days:", np.median(inches[rainy]))
print("Max precip on summer days  :", np.max(inches[summer]))
print("Non-summer rainy median    :", np.median(inches[rainy & ~summer]))

# --- and/or vs &/| ---
A = np.array([1, 0, 1, 0, 1, 0], dtype=bool)
B = np.array([1, 1, 1, 0, 1, 1], dtype=bool)
print(A | B)                     # element-wise OR -> works
# print(A or B)                  # would raise ValueError (ambiguous truth value)

# bitwise on integers operates on bits
print(bin(42 & 59))              # 0b101010
print(bool(42 and 0), bool(42 or 0))  # whole-object truth: False True

## Why this matters / intuition

Masking replaces explicit Python loops with vectorized, C-speed operations. Because a comparison produces a Boolean array that lines up element-for-element with the data, you can ask aggregate questions ("how many?", "any?", "all?") and extract conditional subsets ("only the rainy days") in one expression. This is the same mental model that powers conditional selection in Pandas later, so it pays off well beyond NumPy. *(added context)* The pattern `data[condition]` is essentially SQL's `WHERE` clause expressed in array syntax.

## Gotchas

- **Use the NumPy versions**, not Python built-ins: prefer `np.sum`, `np.any`, `np.all` over `sum`, `any`, `all`, which can be slow or give wrong results on arrays.
- **Parentheses are mandatory** around each compound condition because `&`/`|` have higher precedence than comparison operators — `inches > 0.5 & inches < 1` parses wrong.
- **Never use `and`/`or` on arrays** — it tries to reduce the whole array to a single truth value and raises `ValueError: The truth value of an array ... is ambiguous`. Use `&`/`|`/`~`.
- `&`/`|` on plain integers do **bitwise** arithmetic on the binary representation, which is a different thing from Boolean array logic even though the operator is the same.

## Suggested figure (optional)

A side-by-side panel: left shows a small numeric array with a comparison condition drawn beneath it producing a same-shaped grid of True/False cells (color-coded); right shows the masking step where only the True-cell values are "pulled out" into a shorter 1D array — visually conveying that `x[mask]` collapses the grid down to the selected entries.

# Notes 07 — Fancy Indexing (Vanderplas Ch. 2.7)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.07-fancy-indexing.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Fancy indexing is just like the simple indexing seen earlier (`x[0]`, `x[:5]`), except we pass **arrays of indices in place of single scalars**. This lets us access and modify many array elements at once. The key rule: **the shape of the result reflects the shape of the *index arrays*, not the shape of the array being indexed.**

## Key ideas / idioms

- Pass a list/array of indices: `x[[3, 7, 4]]` grabs three elements in one shot.
- Result shape follows the index shape:
$$\text{shape}(x[\mathrm{ind}]) = \text{shape}(\mathrm{ind})$$
- In multiple dimensions, the first index array selects rows, the second selects columns; they are **paired** element-wise: `X[row, col]` gives `[X[row[0],col[0]], X[row[1],col[1]], ...]`.
- Index arrays combine under **broadcasting rules** — a column vector of rows against a row vector of columns produces a 2D result.
- Fancy indexing mixes with the other indexing styles: simple indices, slices, and boolean masks.
- Fancy indices can be used on the **left-hand side** of an assignment to modify values.
- Repeated indices with augmented assignment (`+=`, etc.) do **not** accumulate — use `np.add.at()` for that.

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- 1D fancy indexing ---
x = np.array([51, 92, 14, 71, 60, 20, 82, 86, 74, 74])
ind = [3, 7, 4]
print(x[ind])              # [71 86 60]

# Result shape mirrors the index shape, not x's shape
ind2 = np.array([[3, 7],
                 [4, 5]])
print(x[ind2])             # [[71 86]
                           #  [60 20]]

# --- Multidimensional fancy indexing (paired) ---
X = np.arange(12).reshape((3, 4))
row = np.array([0, 1, 2])
col = np.array([2, 1, 3])
print(X[row, col])         # [ 2  5 11]

# Broadcasting of index arrays -> 2D result
print(X[row[:, np.newaxis], col])
# Each row index paired with each col index:
# [[ 2  1  3]
#  [ 6  5  7]
#  [10  9 11]]

# --- Combined indexing ---
print(X[2, [2, 0, 1]])     # simple + fancy -> [10  8  9]
print(X[1:, [2, 0, 1]])    # slice + fancy
mask = np.array([1, 0, 1, 0], dtype=bool)
print(X[row[:, np.newaxis], mask])   # fancy + boolean mask

# --- Modifying values ---
x = np.arange(10)
i = np.array([2, 1, 8, 4])
x[i] = 99
print(x)                   # [ 0 99 99  3 99  5  6  7 99  9]
x[i] -= 10
print(x)                   # [ 0 89 89  3 89  5  6  7 89  9]

# Gotcha: repeated indices do NOT accumulate
x = np.zeros(10)
x[[0, 0]] = [4, 6]         # 6 wins (last assignment), not 10
print(x[0])                # 6.0

i = [2, 3, 3, 4, 4, 4]
x = np.zeros(10)
x[i] += 1                  # each slot incremented once, not by repeat count
print(x)                   # index 3 and 4 are only 1.0

# --- np.add.at() does true repeated accumulation ---
x = np.zeros(10)
np.add.at(x, i, 1)
print(x)   # [0. 0. 1. 2. 3. 0. 0. 0. 0. 0.]

# --- Binning data by hand (manual histogram) ---
np.random.seed(42)
x = np.random.randn(100)
bins = np.linspace(-5, 5, 20)
counts = np.zeros_like(bins)
idx = np.searchsorted(bins, x)   # which bin each point falls into
np.add.at(counts, idx, 1)        # accumulate counts
print(counts)

## Why this matters / intuition

Fancy indexing is the building block that lets you express "gather these specific, scattered elements" or "scatter these values into these specific slots" as a single vectorized operation — no Python loop. Once you understand it together with `np.add.at()` and `np.searchsorted()`, you can hand-build specialized routines (like the binning/histogram above) when NumPy's convenience functions don't precisely match your need. For small datasets the hand-rolled version can even beat `np.histogram()`, though the built-in optimizes better for large arrays.

## Gotchas

- **Result shape = index shape**, not array shape. A 2D index array yields a 2D result even from a 1D array.
- **Pairing, not cross-product**, for `X[row, col]` unless you deliberately broadcast (e.g. `row[:, np.newaxis]`).
- **Repeated indices with `+=`/`x[i] += 1` do not accumulate.** Augmented assignment is `x[i] = x[i] + 1` evaluated once — repeats are silently lost. Use `np.add.at(x, i, 1)` for true per-occurrence accumulation.
- Plain assignment with duplicate indices keeps only the **last** write.

## Suggested figure (optional)

A side-by-side diagram of the manual binning pipeline: a strip of sample points along a number line, vertical bin edges from `np.linspace`, arrows from each point to its `searchsorted` bin, and the resulting `counts` bar chart — visually showing how `np.add.at` scatters each point into its bin to form a histogram.

---

## 💬 Q&A (captured during session)

### Q: What is fancy indexing?

**Fancy indexing** means passing an **array (or list) of indices** to grab *multiple* arbitrary elements at once — instead of a single index or a contiguous slice. The result takes the **shape of the index array**, not the shape of the array being indexed.

**The three ways to index, compared:**

In [ ]:
import numpy as np
x = np.array([10, 20, 30, 40, 50, 60])

x[2]            # simple    -> 30           (one element)
x[1:4]          # slicing   -> [20 30 40]   (a contiguous run)
x[[0, 2, 5]]    # fancy     -> [10 30 60]   (any elements you list)

That last one is fancy indexing: hand it a list of positions and it pulls exactly those, in that order, with repeats allowed (`x[[0, 0, 5]]` → `[10 10 60]`).

**Result shape follows the *index* shape:**

In [ ]:
ind = np.array([[0, 1],
                [2, 5]])
x[ind]          # -> [[10 20]
                #     [30 60]]   <- 2x2, same shape as ind

**Multiple dimensions (indices pair up):**

In [ ]:
M = np.arange(12).reshape(3, 4)
rows = np.array([0, 1, 2])
cols = np.array([1, 3, 0])
M[rows, cols]   # -> [1, 7, 8]   picks (0,1), (1,3), (2,0)

Row and column index arrays are matched element-by-element (and broadcast together if shapes differ).

**Assignment works too:**

In [ ]:
x = np.zeros(5)
x[[0, 2, 4]] = 9
x               # -> [9. 0. 9. 0. 9.]

**Why it matters / the gotcha:** fancy indexing is how you **select, reorder, or scatter** arbitrary elements in one vectorized shot — sampling specific rows, reordering by `argsort`, building histograms. One trap: repeated indices in an augmented assignment like `x[[0, 0]] += 1` only add **once** (it's "fetch → assign," not accumulate) — use `np.add.at(x, [0, 0], 1)` for true accumulation.

### Q: What is seeding, and what does `RandomState` do?

Computers don't make true randomness — they compute **pseudo-random** numbers with a formula
that needs a starting input. That starting input is the **seed**, and "seeding" means setting
it. From one seed the formula cranks out a long stream that *looks* random but is fully
determined:

```
seed ─► formula ─► n1 ─► formula ─► n2 ─► formula ─► n3 ─► ...
```

Same seed → identical sequence every run (**reproducible**); different seed → a different but
equally fixed sequence.

In [ ]:
import numpy as np
np.random.seed(42); print(np.random.randint(0, 10, 5))   # [6 3 7 4 6]
np.random.seed(42); print(np.random.randint(0, 10, 5))   # [6 3 7 4 6]  ← repeat
np.random.seed(7);  print(np.random.randint(0, 10, 5))   # [4 9 6 3 3]  ← different

**`np.random.RandomState(seed)`** bakes a seed into your **own private generator object** —
an independent dice-roller, isolated from the global one so other code can't disturb its
sequence. That's why Vanderplas writes `rand = np.random.RandomState(42)`: the book's "random"
arrays come out identical for every reader.

In [ ]:
np.random.seed(42)                  # seeds the shared GLOBAL generator
rand = np.random.RandomState(42)    # your OWN independent, seeded generator
rand.randint(0, 10, 5)              # draws only from yours

- `np.random.randint(...)` (no object) → draws from one hidden **global** generator shared
  program-wide; anyone calling `np.random.seed(...)` elsewhere can change your results.
- `np.random.RandomState(42)` → a **private** generator you control.
- *(added context)* Modern NumPy (≥1.17) prefers `rng = np.random.default_rng(42)`, a newer
  `Generator` with a better algorithm. Same idea; the book predates it.

**One line:** seeding fixes the generator's starting number so its "random" stream is exactly
repeatable; `RandomState`/`default_rng` give you a private, seeded generator to do it cleanly.

# Notes 08 — Sorting Arrays (Vanderplas Ch. 2.8)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.08-sorting.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
NumPy ships fast, compiled sorting routines that vastly outperform hand-written Python sorts. The two workhorses are `np.sort` (returns sorted values) and `np.argsort` (returns the *indices* that would sort the array). Both can sort along a chosen `axis` of a multidimensional array. When you don't need a full sort — only the *k* smallest elements — `np.partition` / `np.argpartition` do the job faster. A capstone example uses these tools to compute k-nearest neighbors via broadcasting.

## Key ideas / idioms
- **Naive sorts are slow.** Selection sort is $$O(N^2)$$ — doubling $N$ roughly quadruples the time. Bogosort (shuffle until sorted) is $$O(N \times N!)$$ — a joke, never use it.
- **`np.sort` is fast.** Uses an introsort/quicksort by default at $$O(N \log N)$$ on average. Returns a *new* array; `x.sort()` sorts *in place*.
- **`np.argsort`** returns indices; `x[np.argsort(x)]` reconstructs the sorted array. Useful when you need the ordering to reindex *related* data.
- **`axis` keyword** sorts each row (`axis=1`) or each column (`axis=0`) independently — relationships across the other axis are lost.
- **Partitioning** (`np.partition(x, k)`) places the $k$ smallest values left of index `k` (in arbitrary order), the rest on the right. Faster than full sort when $k \ll N$. `np.argpartition` gives the indices.
- **Big-O intuition:** scaling matters most at huge $N$; on small arrays a worse-scaling algorithm can still win in wall-clock time.

## Worked code examples (runnable)

In [ ]:
import numpy as np

# --- Naive selection sort (O(N^2)) for intuition ---
def selection_sort(x):
    for i in range(len(x)):
        swap = i + np.argmin(x[i:])
        (x[i], x[swap]) = (x[swap], x[i])
    return x

print(selection_sort(np.array([2, 1, 4, 3, 5])))  # [1 2 3 4 5]

In [ ]:
import numpy as np

# --- np.sort vs in-place sort ---
x = np.array([2, 1, 4, 3, 5])
print(np.sort(x))   # [1 2 3 4 5]  (new array)
print(x)            # [2 1 4 3 5]  (unchanged)

x.sort()            # in-place
print(x)            # [1 2 3 4 5]

# --- np.argsort: indices that would sort the array ---
x = np.array([2, 1, 4, 3, 5])
i = np.argsort(x)
print(i)            # [1 0 3 2 4]
print(x[i])         # [1 2 3 4 5]  (fancy-index with the order)

In [ ]:
import numpy as np

# --- Sorting along an axis ---
rand = np.random.RandomState(42)
X = rand.randint(0, 10, (4, 6))
print(X)
print(np.sort(X, axis=0))  # sort each column independently
print(np.sort(X, axis=1))  # sort each row independently

In [ ]:
import numpy as np

# --- Partial sort: k smallest with np.partition ---
x = np.array([7, 2, 3, 1, 6, 5, 4])
print(np.partition(x, 3))  # 3 smallest left of index 3, e.g. [2 1 3 4 6 5 7]

# argpartition along rows of a 2D array
rand = np.random.RandomState(42)
X = rand.randint(0, 10, (4, 6))
print(np.argpartition(X, 2, axis=1))

In [ ]:
import numpy as np

# --- k-Nearest Neighbors via broadcasting + sorting ---
rand = np.random.RandomState(42)
X = rand.rand(10, 2)

# pairwise squared distances (10x10) via broadcasting
dist_sq = np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=-1)

# full sort: column 0 of each row is the point itself (distance 0)
nearest = np.argsort(dist_sq, axis=1)
print(nearest)

# efficient: only partition out the K+1 nearest (self + K neighbors)
K = 2
nearest_partition = np.argpartition(dist_sq, K + 1, axis=1)
print(nearest_partition[:, :K + 1])

## Why this matters / intuition
Sorting and "find the k smallest" are everywhere in data science: ranking, top-k retrieval, nearest-neighbor search, and feature selection. Knowing that `argsort`/`argpartition` return *indices* lets you reorder one array by another's values — the foundation of vectorized KNN. Choosing `partition` over a full `sort` when you only need the closest few can be a large speedup at scale. *(added context: this partition-for-top-k pattern is exactly what production KNN/ANN libraries optimize.)*

## Gotchas
- `np.sort(x)` returns a copy; `x.sort()` mutates in place — don't confuse the two.
- Sorting with `axis` treats each row/column as *independent*; you destroy cross-axis row alignment. To keep rows intact while ordering by a key column, use `argsort` on that column and fancy-index the whole array. *(added context)*
- `np.partition` does **not** sort: the $k$ smallest are merely on the left in arbitrary order. Don't assume `result[:k]` is itself sorted.
- In the KNN example, each point's nearest "neighbor" is itself (distance 0), so use `K + 1` and drop the first column.
- Better big-O does not guarantee faster runtime on small inputs — measure when in doubt.

## Suggested figure (optional)
A 2D scatter of the 10 KNN points with thin line segments drawn from each point to its 2 nearest neighbors, visually confirming that the partition-selected indices correspond to the geometrically closest points.

---

## 💬 Q&A (captured during session)

### Q: What is the `argsort` function?

**`np.argsort`** ("argument sort") returns the **indices that would sort the array** — the positions to pick, in order, to arrange the array smallest → largest — instead of the sorted values themselves.

**Sort vs argsort:**

In [ ]:
import numpy as np
x = np.array([30, 10, 50, 20])

np.sort(x)      # [10 20 30 50]   <- the sorted VALUES
np.argsort(x)   # [1  3  0  2]    <- the INDICES that sort it

Read `[1 3 0 2]` as: *"smallest value is at index 1, next at index 3, then 0, then 2."*

**Feed it back as fancy indexing** to reconstruct the sorted array:

In [ ]:
order = np.argsort(x)   # [1 3 0 2]
x[order]                # [10 20 30 50]  -> same as np.sort(x)

**Why use argsort instead of sort** — the index order lets you reorder *other* arrays the same way (sort one column, carry the rest along):

In [ ]:
names  = np.array(["Cara", "Al", "Eve", "Bob"])
scores = np.array([30, 10, 50, 20])

order = np.argsort(scores)     # [1 3 0 2]  (lowest score first)
names[order]                   # ['Al' 'Bob' 'Cara' 'Eve']
scores[order]                  # [10 20 30 50]

**Companions:**

In [ ]:
np.argsort(x)[::-1]          # descending (reverse the index list)
np.argmin(x), np.argmax(x)   # index of just the min / max
np.argsort(M, axis=1)        # row-wise index sort for a 2-D array

**Mental model:** `sort` answers *"what are the values in order?"*; `argsort` answers *"where do I find them, in order?"* — more useful whenever positions matter (reordering companions, top-k, ranking).

# Notes 09 — Structured Data: NumPy's Structured Arrays (Vanderplas Ch. 2.9)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/02.09-structured-data-numpy.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Structured arrays let a single NumPy array hold **heterogeneous, compound records** — fields of different types (e.g., a string name, an int age, a float weight) packed together with named fields. They solve the problem of keeping related data spread across separate parallel arrays (where the relationship between entries is only implicit). For everyday work the book notes that **Pandas `DataFrame`s** are usually the better tool; structured arrays shine mainly for simple operations and for mapping onto C/Fortran binary layouts.

## Key ideas / idioms
- A structured array is defined by a compound **`dtype`** listing field names and per-field formats.
- Several equivalent ways to specify the dtype: dictionary of `names`/`formats`, list of `(name, format)` tuples, or a comma-separated string (which auto-names fields `f0, f1, ...`).
- Format codes combine a **type character** + **byte size** (and optional endianness prefix `<`/`>`):
  - `'U10'` = Unicode string up to 10 chars; `'i4'` = 32-bit int; `'f8'` = 64-bit float; `'S10'` = byte string.
- Access patterns:
  - by **field name**: `data['name']` returns that column as an array;
  - by **index**: `data[0]` returns one record (a tuple-like row);
  - **chained**: `data[-1]['name']`;
  - **boolean masking on a field**: `data[data['age'] < 30]['name']`.
- Fields can themselves be sub-arrays: e.g. a `(3, 3)` matrix per record — this maps directly onto C `struct` definitions.
- **`np.recarray`** is a view that adds **attribute-style** access (`data_rec.age`) at the cost of speed.

Format-code reference table from the section:

| Char | Meaning | Example |
|------|---------|---------|
| `'b'` | Byte | `np.dtype('b')` |
| `'i'` | Signed integer | `np.dtype('i4') == np.int32` |
| `'u'` | Unsigned integer | `np.dtype('u1') == np.uint8` |
| `'f'` | Floating point | `np.dtype('f8') == np.float64` |
| `'c'` | Complex float | `np.dtype('c16') == np.complex128` |
| `'S'`, `'a'` | String | `np.dtype('S5')` |
| `'U'` | Unicode string | `np.dtype('U') == np.str_` |
| `'V'` | Raw (void) data | `np.dtype('V') == np.void` |

*(added context)* Per-field access is cheap because each field is a strided view into the same contiguous buffer — no copy is made.

## Worked code examples (runnable)

In [ ]:
import numpy as np

# The "before" problem: parallel lists with only implicit relationships
name = ['Alice', 'Bob', 'Cathy', 'Doug']
age = [25, 45, 37, 19]
weight = [55.0, 85.5, 68.0, 61.5]

# Create a structured array via the dictionary spec
data = np.zeros(4, dtype={'names': ('name', 'age', 'weight'),
                          'formats': ('U10', 'i4', 'f8')})
print(data.dtype)
# [('name', '<U10'), ('age', '<i4'), ('weight', '<f8')]

# Fill the named fields
data['name'] = name
data['age'] = age
data['weight'] = weight
print(data)

# Access by field name (returns the whole column)
print(data['name'])

# Access a single record by index, and a field of it
print(data[0])
print(data[-1]['name'])

# Boolean filtering on a field: names of everyone under 30
print(data[data['age'] < 30]['name'])

In [ ]:
import numpy as np

# Equivalent dtype specifications
d1 = np.dtype({'names': ('name', 'age', 'weight'),
               'formats': ((np.str_, 10), int, np.float32)})  # Python types
d2 = np.dtype([('name', 'S10'), ('age', 'i4'), ('weight', 'f8')])  # list of tuples
d3 = np.dtype('S10,i4,f8')  # comma string -> auto field names f0, f1, f2
print(d1)
print(d2)
print(d3)

In [ ]:
import numpy as np

# Compound type with a per-record sub-array (3x3 matrix)
tp = np.dtype([('id', 'i8'), ('mat', 'f8', (3, 3))])
X = np.zeros(1, dtype=tp)
print(X[0])          # (0, [[0,...]])  id plus a 3x3 block
print(X['mat'][0])   # the 3x3 matrix for record 0

In [ ]:
import numpy as np

data = np.zeros(4, dtype={'names': ('name', 'age', 'weight'),
                          'formats': ('U10', 'i4', 'f8')})
data['age'] = [25, 45, 37, 19]

# Record array: attribute-style access
data_rec = data.view(np.recarray)
print(data_rec.age)   # array([25, 45, 37, 19], dtype=int32)

## Q&A capture — "what *is* a structured array?" (plain version)

> *(My question during reading: just tell me what it is, simply.)*

A normal NumPy array is **homogeneous** — every slot holds **one value** of **one type**
(all ints, all floats). A **structured array** lets every slot instead hold a **record**: a
bundle of several **named fields**, each with its own type. Each element becomes a mini-row of
a table.

In [ ]:
data[0]        # ('Alice', 25, 55.0)  ← one record = labeled bundle of mixed types
data['name']   # whole 'name' column as an array

The job it does: instead of three fragile **parallel lists** (where "Alice/25/55.0" is linked
only by all sharing index 0, and sorting one breaks the link), it **fuses them into one
container** so each record stays glued together.

```
  normal array        structured array
  ┌────┐              ┌──────────────────────────────┐
  │ 25 │ one value    │ name='Alice' age=25 wt=55.0  │ one record
  │ 45 │ per slot     │ name='Bob'   age=45 wt=85.5  │ (named fields, mixed types)
  └────┘              └──────────────────────────────┘
```

The compound **`dtype`** is what makes it work — it names each field and its type+size
(`'U10'` = 10-char string, `'i4'` = 32-bit int, `'f8'` = 64-bit float), laid out back-to-back
in memory. **One line:** a structured array is a NumPy array whose every element is a labeled,
mixed-type record — a whole table packed into one contiguous buffer. (For daily work, reach for
a Pandas `DataFrame`; structured arrays earn their keep mainly for C/Fortran binary layouts.)

## Why this matters / intuition
Structured arrays give you a **single, self-describing container** for tabular/record data while keeping NumPy's contiguous-memory efficiency. Because the layout is explicit (named fields, fixed sizes, endianness), they are ideal for **interoperating with C/Fortran binary formats and legacy libraries** — the memory image lines up with a C `struct`. They are also the conceptual ancestor of the Pandas `DataFrame`, so understanding them clarifies how Pandas stores typed columns under the hood.

## Gotchas
- **`recarray` attribute access is slow.** The book's timings: `data['age']` ≈ 241 ns; `data_rec['age']` ≈ 4.61 µs; `data_rec.age` ≈ 7.27 µs. The attribute convenience adds real overhead — prefer dict-key access in hot loops.
- The comma-string spec (`'S10,i4,f8'`) gives you **no real field names** (`f0, f1, f2`); use it only when names don't matter.
- `'S'` is a **byte string** while `'U'` is **Unicode** — mixing them up can cause encoding surprises *(added context: `'S'` fields compare against `bytes`, not `str`)*.
- For routine data wrangling, reach for **Pandas** instead; structured arrays are best for simple ops and C-layout mapping.

## Suggested figure (optional)
A side-by-side diagram: on the left, three separate boxes labeled `name`, `age`, `weight` with dashed lines hinting at the implicit row correspondence; on the right, a single contiguous memory strip divided into records, each record sub-divided into colored slots (`U10` | `i4` | `f8`) with a header naming the fields — visually showing how a structured array fuses the parallel arrays into one typed buffer.

# Notes 10 — Introducing Pandas Objects (Vanderplas Ch. 3.1)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Pandas builds on NumPy by attaching **explicit, labeled indices** to data. Three core
objects power everything that follows:

- **`Series`** — a one-dimensional array of *indexed* data. Like a NumPy 1D array, but with
  an explicit index (not just implicit integer positions). Also acts like a typed dictionary.
- **`DataFrame`** — a two-dimensional structure with both flexible row indices and flexible
  column names. Think of it as a generalized NumPy 2D array, or as a dict mapping column
  names to `Series`.
- **`Index`** — the immutable, ordered-set object that labels the axes of `Series`/`DataFrame`.

The unifying idea: NumPy gives you *implicit* integer indexing; Pandas gives you *explicit*
label-based indexing on top of NumPy arrays.

## Key ideas / idioms

- A `Series` has `.values` (a NumPy array) and `.index` (a `pd.Index` object).
- NumPy array → *implicitly* defined integer index. `Series` → *explicitly* defined index,
  which can be any data type (strings, non-sequential ints, etc.).
- A `Series` is like a dictionary mapping typed keys → typed values, but it also supports
  array-style slicing that plain dicts do not.
- General constructor: `pd.Series(data, index=index)`. `data` may be a list/array, a scalar
  (broadcast to fill the index), or a dict (keys become the index).
- A `DataFrame` maps a **column name → a `Series` of column data**. Crucially, `data['col']`
  returns a *column* (a `Series`), whereas NumPy `arr[i]` returns a *row*.
- An `Index` behaves like an immutable array (supports slicing, has `.size/.shape/.ndim/.dtype`)
  and like an ordered set (intersection, union, symmetric difference).

Set operations on indices (the conceptual algebra):

$$
A \cap B \;\;(\text{intersection}), \qquad
A \cup B \;\;(\text{union}), \qquad
A \,\triangle\, B \;\;(\text{symmetric difference})
$$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- Series: basic construction ---
data = pd.Series([0.25, 0.5, 0.75, 1.0])
print(data.values)   # NumPy array: [0.25 0.5  0.75 1.  ]
print(data.index)    # RangeIndex(start=0, stop=4, step=1)

# --- Series as generalized NumPy array: explicit index ---
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
print(data['b'])     # 0.5

# Non-sequential / non-integer indices also work
data2 = pd.Series([0.25, 0.5, 0.75, 1.0], index=[2, 5, 3, 7])
print(data2[5])      # 0.5

In [ ]:
import numpy as np
import pandas as pd

# --- Series as a specialized dictionary ---
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135}
population = pd.Series(population_dict)
print(population['California'])           # dict-style access -> 38332521
print(population['California':'Florida']) # array-style slicing (dicts can't do this)

# --- Constructor variations ---
print(pd.Series([2, 4, 6]))                       # default integer index
print(pd.Series(5, index=[100, 200, 300]))        # scalar broadcast to fill index
print(pd.Series({2: 'a', 1: 'b', 3: 'c'}))        # dict -> index from keys
print(pd.Series({2: 'a', 1: 'b', 3: 'c'}, index=[3, 2]))  # index selects/filters keys

In [ ]:
import numpy as np
import pandas as pd

population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127, 'Florida': 19552860,
                        'Illinois': 12882135})
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})

# --- DataFrame from a dict of Series ---
states = pd.DataFrame({'population': population, 'area': area})
print(states.index)     # row labels (state names)
print(states.columns)   # column labels: ['population', 'area']

# DataFrame as specialized dictionary: column access returns a Series
print(states['area'])   # the 'area' column

# --- Other ways to build a DataFrame ---
# From a single Series
print(pd.DataFrame(population, columns=['population']))

# From a list of dicts (missing keys -> NaN)
print(pd.DataFrame([{'a': i, 'b': 2 * i} for i in range(3)]))
print(pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}]))  # NaN-filled

# From a 2D NumPy array
print(pd.DataFrame(np.random.rand(3, 2),
                   columns=['foo', 'bar'],
                   index=['a', 'b', 'c']))

# From a NumPy structured array
A = np.zeros(3, dtype=[('A', 'i8'), ('B', 'f8')])
print(pd.DataFrame(A))

In [ ]:
import numpy as np
import pandas as pd

# --- The Index object: immutable array ---
ind = pd.Index([2, 3, 5, 7, 11])
print(ind[1])     # 3
print(ind[::2])   # every other element -> [2, 5, 11]
print(ind.size, ind.shape, ind.ndim, ind.dtype)   # 5 (5,) 1 int64

# Immutability: this raises TypeError (indices cannot be modified)
try:
    ind[1] = 0
except TypeError as e:
    print("TypeError:", e)

# --- Index as an ordered set ---
indA = pd.Index([1, 3, 5, 7, 9])
indB = pd.Index([2, 3, 5, 7, 11])
print(indA.intersection(indB))         # [3, 5, 7]
print(indA.union(indB))                # [1, 2, 3, 5, 7, 9, 11]
print(indA.symmetric_difference(indB)) # [1, 2, 9, 11]

## Q&A capture — "generalization of an array" vs "specialization of a dict"

> *(My question during reading: what do those two phrases actually mean?)*

The two phrases compare a `Series` in **opposite directions**:

- **Generalization of a NumPy array** = *more capable / fewer restrictions*. A `Series`
  does everything an array does (typed, contiguous values; vectorized math) **plus** it lets
  you choose the index. A NumPy array is just the special case where the index is forced to be
  `0, 1, 2, …`. Relaxing that "labels must be integer positions" rule = generalizing.

In [ ]:
arr = np.array([0.25, 0.5, 0.75, 1.0]); arr[1]          # 0.5 (label = fixed position)
s = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a','b','c','d']); s['b']  # 0.5 (chosen label)

- **Specialization of a Python dict** = *less flexible, but better at a narrower job*. A dict
  maps arbitrary keys → arbitrary values with no type rules. A `Series` is dict-like
  (label → value) but **constrains** keys to a typed ordered `Index` and values to a
  single-dtype NumPy array. Giving up the dict's "anything goes" freedom buys vectorized math
  and array speed.

In [ ]:
d = {'a':0.25,'b':0.5,'c':0.75}; d['b']   # 0.5, but d*2 / d.mean() are impossible
s = pd.Series(d); s['b']; s * 2; s['a':'c']  # dict lookup AND math AND slicing

A `Series` sits between the two: **add labels to an array** (generalize upward) and
**add array-speed + type rules to a dict** (specialize downward) — getting the best of each.

```
  MORE GENERAL  ◄──────── Series ────────►  MORE SPECIALIZED
  NumPy array            (labels + typed     Python dict
  (fast, but labels       array values)      (label→value, but no
   forced 0,1,2,…)                            math, no type rules)
```

## Q&A capture — "a DataFrame is a sequence of aligned Series"

> *(My question during reading: simplify the book's DataFrame-as-aligned-Series passage.)*

Sentence-by-sentence translation of the source passage:

1. **"DataFrame = 2D array with flexible row indices AND flexible column names."** Same idea
   as a `Series` (1D array with chosen labels) but in two dimensions — you label *both* the
   rows (`'California'`, `'Texas'`, …) and the columns (`'population'`, `'area'`).
2. **"A DataFrame is a sequence of aligned Series."** Picture a 2D array as columns sitting
   side by side; in a `DataFrame` each of those columns *is* a `Series`. Glue several `Series`
   together as columns → a `DataFrame`.
3. **"Aligned" = the Series share the same index.** When stacked, Pandas matches rows by
   **label**, not by position. So `population['California']` and `area['California']` land on
   the same row even if the two Series were in different orders; a label present in one but not
   the other becomes `NaN`.

In [ ]:
population = pd.Series({'California': 38332521, 'Texas': 26448193, 'New York': 19651127})
area2      = pd.Series({'Texas': 695662, 'California': 423967})   # different order, no NY
pd.DataFrame({'population': population, 'area': area2})
#             population      area
# California    38332521  423967.0
# New York      19651127       NaN   ← aligned by label; NY has no area → NaN
# Texas         26448193  695662.0

```
  Series population   Series area        DataFrame (glued on shared index)
  ┌──────────┬───┐   ┌──────────┬─────┐   ┌──────────┬──────────┬──────┐
  │California │38M│   │California │423K │   │          │population│ area │ ← col names
  │New York   │19M│ + │New York   │141K │ = │California │  38M     │423K  │
  │Texas      │26M│   │Texas      │696K │   │New York   │  19M     │141K  │ ← row index
  └──────────┴───┘   └──────────┴─────┘   │Texas      │  26M     │696K  │
       same row labels  =  "aligned"       └──────────┴──────────┴──────┘
```

**One line:** a `DataFrame` is several `Series` standing shoulder to shoulder as columns,
lined up so equal-labeled rows sit on the same line.

## Why this matters / intuition

The explicit index is what makes Pandas powerful: data **aligns by label**, not by position.
When you combine two `Series` or build a `DataFrame` from several `Series`, Pandas matches on
the index automatically and fills mismatches with `NaN`. That label-based alignment is the
seed of nearly every later operation (indexing, joins, groupby, time series). Treating a
`Series` simultaneously as "array" and "dict" gives you both fast vectorized math and
meaningful keyed lookups in one object.

## Gotchas

- **Column vs. row access:** for a `DataFrame`, `data['col0']` returns the first *column*,
  not the first row — the opposite of `arr[0]` on a NumPy 2D array.
- **Indices are immutable:** assigning into an `Index` raises
  `TypeError: Index does not support mutable operations`. This immutability is what makes it
  safe to share an index across multiple `Series`/`DataFrame` objects.
- **Dict construction and ordering:** building a `Series`/`DataFrame` from a dict derives the
  index from the keys; passing an explicit `index=` selects/filters which keys are kept.
- **List-of-dicts with missing keys** produces `NaN` for absent entries — don't assume rows
  are dense.
- *(added context)* In older pandas the set operators `&`, `|`, `^` worked directly on `Index`
  objects (as the book originally showed); current pandas reserves those for elementwise
  boolean ops, so prefer the explicit `.intersection()` / `.union()` /
  `.symmetric_difference()` methods used above.

## Suggested figure (optional)

A three-panel diagram: (1) a NumPy 1D array with implicit integer positions 0–3 beside a
`Series` showing the same values bound to an explicit labeled index (a, b, c, d); (2) a
`DataFrame` drawn as a grid with labeled row index down the left and column names across the
top, each column highlighted as its own `Series`; (3) two overlapping circles (a Venn diagram)
illustrating `Index` set operations — intersection, union, and symmetric difference.

# Notes 11 — Data Indexing and Selection (Vanderplas Ch. 3.2)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.02-data-indexing-and-selection.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Pandas `Series` and `DataFrame` objects extend NumPy-style access (slicing, masking, fancy indexing) while also behaving like dictionaries (key-based access). The friction point is that a `Series`/`DataFrame` carries an **explicit index** that may or may not match the **implicit integer position**. To remove ambiguity, Pandas provides the `loc`, `iloc`, and `ix` indexers. The chapter walks through these patterns for `Series` first, then `DataFrame`.

## Key ideas / idioms

**Two indices, always.** Every Pandas object has an *explicit* index (the labels you assigned) and an *implicit* Python-style integer position. Plain `[]` mixes them in a confusing way:
- `data[1]` → uses the **explicit** index (label lookup).
- `data[1:3]` → uses the **implicit** position (Python-style slicing).

**The disambiguating indexers:**
- `loc` → **always explicit** index/labels. Slices are **inclusive** of the endpoint: `data.loc[1:3]` includes label `3`.
- `iloc` → **always implicit** integer position. Slices are **exclusive** of the endpoint (standard Python): `data.iloc[1:3]` excludes position `3`.
- `ix` → hybrid of the two; for a `Series` it behaves like plain `[]`. *(added context: `ix` is deprecated in modern Pandas — prefer `loc`/`iloc`. The book predates its removal but already recommends avoiding it.)*

**Slicing endpoint rule (the core gotcha), expressed as index sets.** For an explicit-label slice `a:c`:
$$ \text{result} = \{\, i : a \le i \le c \,\} \quad\text{(endpoint included)} $$
For an implicit-position slice `j:k`:
$$ \text{result} = \{\, p : j \le p < k \,\} \quad\text{(endpoint excluded)} $$

**DataFrame access conventions (the "seemingly inconsistent" rules):**
- **Indexing** (single key) refers to **columns**: `data['area']`.
- **Slicing** refers to **rows**: `data['Florida':'Illinois']` or `data[1:3]`.
- **Masking** operates **row-wise**: `data[data.density > 100]`.

**Zen of Pandas guidance:** prefer `loc`/`iloc` explicitly for readable code and to avoid subtle integer-index bugs.

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- Series as dictionary ---
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
print(data['b'])                 # 0.5
print('a' in data)               # True
print(list(data.keys()))         # ['a', 'b', 'c', 'd']
print(list(data.items()))        # [('a', 0.25), ('b', 0.5), ...]

data['e'] = 1.25                 # extend like a dict
print(data)

In [ ]:
import numpy as np
import pandas as pd

# --- Series as 1D array: slice / mask / fancy ---
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])

print(data['a':'c'])             # explicit slice: INCLUDES 'c'
print(data[0:2])                 # implicit slice: EXCLUDES position 2
print(data[(data > 0.3) & (data < 0.8)])   # masking
print(data[['a', 'd']])          # fancy indexing

In [ ]:
import numpy as np
import pandas as pd

# --- The trap: explicit integer index ---
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])

print(data[1])      # explicit index -> 'a'
print(data[1:3])    # implicit position -> 'b','c' (labels 3,5)

# Disambiguate:
print(data.loc[1])      # explicit -> 'a'
print(data.loc[1:3])    # explicit, inclusive -> labels 1 and 3
print(data.iloc[1])     # position 1 -> 'b'
print(data.iloc[1:3])   # positions 1,2 -> 'b','c'

In [ ]:
import numpy as np
import pandas as pd

# --- DataFrame indexing ---
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area': area, 'pop': pop})

print(data['area'])              # column access (dict-style)
print(data.area)                 # column access (attribute-style, same object)
print(data.area is data['area']) # True

data['density'] = data['pop'] / data['area']   # computed column
print(data)

print(data.values)               # underlying NumPy array
print(data.T)                    # transpose

In [ ]:
import numpy as np
import pandas as pd

area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area': area, 'pop': pop})
data['density'] = data['pop'] / data['area']

# --- loc / iloc on DataFrame ---
print(data.iloc[:3, :2])             # first 3 rows, first 2 cols (positional)
print(data.loc[:'Illinois', :'pop']) # label-based, inclusive

# combined masking + fancy indexing
print(data.loc[data.density > 100, ['pop', 'density']])

# assignment via indexer
data.iloc[0, 2] = 90
print(data)

In [ ]:
import numpy as np
import pandas as pd

area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area': area, 'pop': pop})
data['density'] = data['pop'] / data['area']

# --- Extra conventions: slicing -> rows, masking -> rows ---
print(data['Florida':'Illinois'])    # slice = ROWS by label
print(data[1:3])                     # slice = ROWS by position
print(data[data.density > 100])      # mask = ROWS

## Why this matters / intuition
Real data analysis is mostly *selecting the right slice of data*: a column, a window of rows, the rows that satisfy a condition. Pandas overloads `[]` to be ergonomic for the common cases (column by name, row slice, boolean mask), but that overloading is exactly what causes confusion once your index is integer-valued. The `loc`/`iloc` split gives you one unambiguous mental model: "am I talking about *labels* or *positions*?" Choosing deliberately makes selection code readable and prevents off-by-one and wrong-axis bugs that silently corrupt downstream analysis.

## Gotchas
- **Inclusive vs exclusive endpoints differ by indexer.** `loc[1:3]` includes `3`; `iloc[1:3]` and plain `data[1:3]` exclude position `3`. *(added context: easy to drop the last row of a label slice if you assume Python semantics.)*
- **Integer explicit index is the danger zone.** With an integer index, `data[1]` is label lookup but `data[1:3]` is positional. Use `loc`/`iloc` to be safe.
- **Attribute access (`data.area`) is fragile.** It fails or misbehaves when a column name collides with a `DataFrame` method/attribute (e.g., `data.pop` is the `pop()` method, not a column). For *assignment*, always use `data['col'] = ...`, never `data.col = ...`.
- **Single key indexes columns; slices index rows.** `data['area']` is a column, but `data['Florida':'Illinois']` is rows — same brackets, different axis.
- **Masking is row-wise**, returning the rows where the condition is True, not a filtered set of columns.

## Suggested figure (optional)
A two-column "decision card": left column "I want LABELS → use `.loc` (endpoint INCLUDED)", right column "I want POSITIONS → use `.iloc` (endpoint EXCLUDED)", with a small table at the bottom mapping plain-`[]` behaviors: single key = column, slice = rows, boolean mask = rows. A tiny number line could illustrate `loc[1:3]` covering 1,2,3 vs `iloc[1:3]` covering positions 1,2.

---

## 💬 Q&A (captured during session)

### Q: Clarify `loc` vs `iloc`.

Every Pandas object carries **two** indexing systems: the **explicit index** (labels you
assigned) and the **implicit position** (`0,1,2,…` where each item sits). Plain `[]` guesses
between them *inconsistently* — that's the trap `loc`/`iloc` exist to kill.

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data[1]      # -> 'a'        plain [] treats 1 as a LABEL
data[1:3]    # -> labels 3,5 plain [] treats 1:3 as POSITIONS  (inconsistent!)

- **`.loc` = by Label** (explicit index); slice endpoint **INCLUDED**.

In [ ]:
data.loc[1]     # 'a'            item whose label is 1
data.loc[1:3]   # labels 1 and 3 (3 included)

- **`.iloc` = by integer position** (`0,1,2,…`); slice endpoint **EXCLUDED** (normal Python).

In [ ]:
data.iloc[1]    # 'b'            item at position 1
data.iloc[1:3]  # 'b','c'        positions 1,2 (3 excluded)

| | `.loc` | `.iloc` |
|---|---|---|
| indexes by | label (explicit) | position (`0,1,2,…`) |
| `[1]` means | item labeled `1` | 2nd item |
| slice endpoint | **included** | **excluded** |
| mnemonic | **l**oc = **l**abel | **i**loc = **i**nteger |

On a DataFrame both take `[rows, cols]`: `data.iloc[:3, :2]` (positional), 
`data.loc[:'Illinois', :'pop']` (label, inclusive), 
`data.loc[data.density > 100, ['pop','density']]` (mask rows + pick cols by label).

**One line:** `.loc` when thinking in **labels** (endpoint included), `.iloc` when thinking in
**positions** (endpoint excluded) — prefer both over bare `[]` so an integer index can't trick you.

# Notes 12 — Operating on Data in Pandas (Vanderplas Ch. 3.3)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.03-operations-in-pandas.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Pandas inherits NumPy's element-wise (ufunc) machinery but adds **label awareness**:

- **Unary ufuncs** (negation, trig, exp, ...) **preserve the index and column labels** of the input.
- **Binary ufuncs** (`+`, `*`, ...) **automatically align indices** before computing, so you operate on data matched by label rather than by position.

This means messy, differently-ordered, or partially-overlapping datasets combine correctly, with non-overlapping entries filled by `NaN` (or a fill value you choose).

## Key ideas / idioms

- **Ufunc index preservation:** `np.exp(ser)`, `np.sin(df * np.pi/4)` return a Series/DataFrame with the same labels.
- **Series alignment:** result index is the **union** of the two input indices; missing labels become `NaN`.
- **DataFrame alignment:** **both rows and columns** are aligned; the result index/columns are the union, sorted, with missing intersections `NaN`.
- **Fill values:** use the object methods (not the operators) to supply `fill_value`, e.g. `A.add(B, fill_value=0)`.
- **DataFrame–Series ops broadcast row-wise by default** (the Series aligns to columns, applied across each row). Use `axis=0` to broadcast column-wise instead.

Operator ⟷ method equivalents:

$$
\begin{array}{ll}
\texttt{+} & \texttt{add()} \\
\texttt{-} & \texttt{sub(), subtract()} \\
\texttt{*} & \texttt{mul(), multiply()} \\
\texttt{/} & \texttt{truediv(), div(), divide()} \\
\texttt{//} & \texttt{floordiv()} \\
\texttt{\%} & \texttt{mod()} \\
\texttt{**} & \texttt{pow()}
\end{array}
$$

Index union (conceptually): for Series $A$ with index $I_A$ and $B$ with index $I_B$,

$$ \text{index}(A \oplus B) = I_A \cup I_B $$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- Ufuncs preserve index / column labels ---
rng = np.random.RandomState(42)

ser = pd.Series(rng.randint(0, 10, 4))
print(np.exp(ser))          # same index 0..3, exp applied elementwise

df = pd.DataFrame(rng.randint(0, 10, (3, 4)), columns=['A', 'B', 'C', 'D'])
print(np.sin(df * np.pi / 4))   # same rows 0..2, columns A..D

In [ ]:
import numpy as np
import pandas as pd

# --- Index alignment in Series (union -> NaN for missing) ---
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')

print(population / area)    # California, Texas have values; Alaska, New York -> NaN

# --- Custom fill value via the method form ---
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
print(A + B)                # index 1,2 computed; 0,3 -> NaN
print(A.add(B, fill_value=0))   # 0->2.0, 1->5.0, 2->9.0, 3->5.0

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.RandomState(42)

# --- Index alignment in DataFrame (rows AND columns aligned) ---
A = pd.DataFrame(rng.randint(0, 20, (2, 2)), columns=list('AB'))
B = pd.DataFrame(rng.randint(0, 10, (3, 3)), columns=list('BAC'))
print(A + B)                # union of rows/cols; non-overlap -> NaN

# Fill missing intersections with the mean of all values in A
fill = A.stack().mean()
print(A.add(B, fill_value=fill))

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.RandomState(42)

# --- DataFrame and Series operations ---
A = rng.randint(10, size=(3, 4))
df = pd.DataFrame(A, columns=list('QRST'))

# Row-wise by default: subtract first row from every row
print(df - df.iloc[0])

# Column-wise: broadcast a column down each row with axis=0
print(df.subtract(df['R'], axis=0))

# Operations also align by index/columns -> NaN where unmatched
halfrow = df.iloc[0, ::2]   # columns Q and S only
print(df - halfrow)         # R and T columns -> NaN

## Why this matters / intuition

Raw NumPy operations match by **position**, so combining datasets that are ordered differently or have different membership silently produces wrong answers. Pandas matches by **label**, so "California population / California area" is computed regardless of row order, and any mismatch shows up loudly as `NaN` instead of a silent misalignment. The data keeps its context through every operation.

## Gotchas

- The **operator** form (`A + B`) cannot take a `fill_value`; you must use the **method** form (`A.add(B, fill_value=...)`).
- Filling with `0` is not always right — for a DataFrame, `A.stack().mean()` (mean of all entries) is one reasonable choice; pick a fill that makes sense for your data.
- `NaN` propagates: any unmatched label/intersection becomes `NaN`, which can then poison later reductions unless handled.
- DataFrame–Series ops are **row-wise by default**; forgetting `axis=0` for a column-wise operation is a common silent bug.
- Result indices/columns of aligned ops are returned **sorted** (union), so order may differ from either input. *(added context)*

## Suggested figure (optional)

A two-panel diagram. Left: two Series drawn as labeled boxes with partially overlapping index labels; arrows show alignment by label and a union index on the output, with `NaN` boxes where a label exists in only one input. Right: a DataFrame grid plus a Series shown once as a row (default broadcast, arrows fanning downward across rows) and once as a column (`axis=0`, arrows fanning rightward across columns), illustrating the two broadcast directions.

---

## 💬 Q&A (captured during session)

### Q: Help me read the arguments in `pd.DataFrame(rng.randint(0, 20, (2, 2)), ...)`.

Read it **inside-out** — two nested calls.

**Inner — `rng.randint(0, 20, (2, 2))`** draws random integers from the seeded generator `rng`:

| Position | Value | Name | Meaning |
|---|---|---|---|
| 1st | `0` | `low` | smallest int it can draw (**inclusive**) |
| 2nd | `20` | `high` | upper bound (**exclusive** → max is `19`) |
| 3rd | `(2, 2)` | `size` | **shape** of the output: 2 rows × 2 cols (one tuple, not two numbers) |

→ a 2×2 NumPy array of ints in `0..19`.

**Outer — `pd.DataFrame(...)`** wraps that array into a labeled table. With no other args, rows
and columns auto-label `0,1`; with `columns=list('AB')` the columns become `'A','B'`:

In [ ]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)), columns=list('AB'))
#     A   B
# 0   3  17
# 1   9   2

```
rng.randint( 0 , 20 , (2,2) )   ->   2x2 array   ->   pd.DataFrame(arr, columns=list('AB'))
            low  high  shape                           wrap as labeled table, name cols A,B
```

**One breath:** "draw a 2×2 grid of random ints 0–19, then wrap it in a DataFrame named `A`."

*(note: a snippet like `pd.DataFrame(rng.randint(0, 20, (2, 2))` with no closing `)` is just
missing a paren — the book line continues with `, columns=...)`.)*
---

# Notes 13 — Handling Missing Data (Vanderplas Ch. 3.4)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.04-missing-values.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Real-world data is rarely clean; missing values must be represented somehow. There are two broad strategies: a **mask** (a separate Boolean array marking missing entries) or a **sentinel value** (a special in-band value that means "missing"). Pandas chose the sentinel approach, leaning on two existing Python/NumPy nulls:

- **`None`** — the Python singleton object. It only works inside arrays of `dtype=object`, which forces operations to run at slow Python-object speed and breaks compiled aggregations.
- **`NaN`** — the IEEE-754 floating-point "Not a Number" value. It lives in fast native `float64` arrays, so vectorized ops stay compiled, but it propagates through arithmetic.

Pandas papers over the differences between these two and handles converting/upcasting between them automatically.

## Key ideas / idioms
- **`None` = Python object sentinel.** An array containing `None` is upcast to `dtype=object`. Aggregations like `.sum()` raise `TypeError` because Python can't add `int + None`.
- **`NaN` = IEEE floating-point sentinel.** It lives in `dtype=float64`. Arithmetic is well-defined but "viral."
- **NaN propagation.** "NaN is a bit like a data virus — it infects any other object it touches." Any operation with NaN yields NaN:
$$ 1 + \texttt{NaN} = \texttt{NaN}, \qquad 0 \times \texttt{NaN} = \texttt{NaN} $$
  So `arr.sum()`, `arr.min()`, `arr.max()` all return `NaN` if any element is NaN. Use NumPy's NaN-aware versions (`np.nansum`, `np.nanmin`, `np.nanmax`) to ignore missing entries.
- **Automatic upcasting** when NA enters a typed array:
  - Floating → no change, sentinel `np.nan`
  - Object → no change, sentinel `None` or `np.nan`
  - Integer → cast to `float64`, sentinel `np.nan`
  - Boolean → cast to `object`, sentinel `None` or `np.nan`
- **Pandas treats `None` and `NaN` as interchangeable** for missing-value purposes and converts between them as needed.
- **Four core methods:** `isnull()` / `notnull()` (detect), `dropna()` (remove), `fillna()` (replace).

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# 1) None forces dtype=object and breaks aggregation
vals1 = np.array([1, None, 3, 4])
print(vals1.dtype)                 # object
try:
    print(vals1.sum())
except TypeError as e:
    print("None array sum -> TypeError:", e)

# 2) NaN lives in float64; aggregations propagate NaN
vals2 = np.array([1, np.nan, 3, 4])
print(vals2.dtype)                 # float64
print(1 + np.nan, 0 * np.nan)      # nan nan
print(vals2.sum(), vals2.min(), vals2.max())   # nan nan nan

# NaN-aware aggregations skip the missing value
print(np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2))  # 8.0 1.0 4.0

In [ ]:
import numpy as np
import pandas as pd

# 3) Pandas upcasts int -> float64 when a null is introduced
x = pd.Series(range(2), dtype=int)
print(x.dtype)        # int64
x[0] = None
print(x.dtype)        # float64  (None became NaN)
print(x)

In [ ]:
import numpy as np
import pandas as pd

# 4) Detecting nulls
data = pd.Series([1, np.nan, 'hello', None])
print(data.isnull())     # True where missing
print(data.notnull())    # opposite mask
print(data[data.notnull()])   # boolean-index out the non-null values

In [ ]:
import numpy as np
import pandas as pd

# 5) Dropping nulls in a Series
data = pd.Series([1, np.nan, 'hello', None])
print(data.dropna())

# DataFrames: cannot drop single cells, only whole rows/columns
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
print(df.dropna())                 # default axis=0: drop any row with NA
print(df.dropna(axis='columns'))   # drop any column with NA
df[3] = np.nan
print(df.dropna(axis='columns', how='all'))   # only drop all-NA columns
print(df.dropna(axis='rows', thresh=3))       # keep rows with >= 3 non-null

In [ ]:
import numpy as np
import pandas as pd

# 6) Filling nulls
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))
print(data.fillna(0))              # replace with a constant
print(data.fillna(method='ffill')) # forward-fill: carry previous value forward
print(data.fillna(method='bfill')) # back-fill: carry next value backward

# DataFrame fills can take an axis (fill direction)
df = pd.DataFrame([[1, np.nan, 2],
                   [2, 3,      5],
                   [np.nan, 4, 6]])
print(df.fillna(method='ffill', axis=1))   # fill across columns

## Why this matters / intuition
Missing data is the rule, not the exception, in real datasets. Understanding *which* sentinel Pandas is using tells you about performance (object dtype = slow Python loop; float64 = fast compiled) and about silent type changes (your integer column quietly becoming float). Knowing that NaN propagates explains why a single missing value can turn an entire `.sum()` into `NaN` — and why the NaN-aware aggregations exist. The `isnull`/`dropna`/`fillna` trio covers the full workflow: detect, then either remove or impute.

## Gotchas
- `np.array([1, None, 3]).sum()` raises `TypeError`, but `np.array([1, np.nan, 3]).sum()` returns `nan` — different failure modes for the two sentinels.
- Introducing a null into an **integer** Series silently upcasts it to **float64**; into a **boolean** Series upcasts to **object**.
- Plain `arr.sum()`/`.min()`/`.max()` return `NaN` if any value is NaN — reach for `np.nansum`, etc., to ignore them.
- On a DataFrame you cannot drop a single missing cell — `dropna()` removes the entire row or column. Use `how`/`thresh` to control how aggressive it is.
- `NaN == NaN` is `False` *(added context: IEEE semantics — never test for missingness with `==`; use `isnull()`)*.
- *(added context: in newer Pandas, `fillna(method='ffill')` is deprecated in favor of `df.ffill()` / `df.bfill()`; the handbook predates this.)*

## Suggested figure (optional)
A two-panel side-by-side diagram. Left panel: an array `[1, None, 3, 4]` boxed as `dtype=object`, with each cell drawn as a separate Python object pointer, and a red "TypeError" stamp over a `sum()` arrow. Right panel: an array `[1, NaN, 3, 4]` boxed as a contiguous `float64` block, with arrows showing NaN "infecting" the running total so `sum() -> NaN`, plus a green branch labeled `np.nansum -> 8.0` that skips the NaN cell. A small legend maps each upcasting rule (int→float64, bool→object).

# Notes 14 — Hierarchical Indexing / MultiIndex (Vanderplas Ch. 3.5)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.05-hierarchical-indexing.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

**Hierarchical indexing** (a.k.a. **multi-indexing**) lets you pack data with more than one or two dimensions into the familiar 1D `Series` and 2D `DataFrame` by giving an axis *multiple index levels*. A single `Series` with a two-level index behaves like a 2D table; a `DataFrame` with multi-level row *and* column indices behaves like 4D data, and so on. This is the idiomatic Pandas way to handle higher-dimensional, labeled data without leaving the core data structures.

## Key ideas / idioms

- A naive approach uses **Python tuples as keys** (e.g. `('California', 2000)`). It "works" but slicing on a single level forces ugly comprehensions and is inefficient.
- `pd.MultiIndex` is the proper object. Pandas often builds it implicitly; you can also build it explicitly with `from_arrays`, `from_tuples`, `from_product`, or the low-level constructor.
- A `Series` with a `MultiIndex` and an `unstack()` of it into a `DataFrame` carry the **same information** — the extra index level is "one more dimension."
- **Index levels can be named** (`index.names`), which makes selection self-documenting.
- **Both axes** (rows and columns) can be hierarchical.
- **Partial indexing / slicing** selects on a subset of levels.
- Many `MultiIndex` slicing operations require the index to be **lexicographically sorted**; otherwise you get an error. Fix with `sort_index()`.
- `stack` / `unstack` reshape between stacked (index) and pivoted (column) representations, controllable by `level`.
- `reset_index` turns index levels into columns; `set_index` does the reverse. This is the bridge between flat tables and multi-indexed ones.
- Aggregations can collapse a chosen `level` instead of the whole axis.

For an $n$-level row index and an $m$-level column index, a `DataFrame` effectively represents data of dimensionality

$$ \text{dims} = n + m $$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- The clumsy "bad way": tuples as keys ---
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956, 18976457, 19378102, 20851820, 25145561]
pop = pd.Series(populations, index=index)

# Selecting one level requires an awkward comprehension:
print(pop[[i for i in pop.index if i[1] == 2010]])

In [ ]:
import numpy as np
import pandas as pd

# --- The better way: a real MultiIndex ---
index = pd.MultiIndex.from_tuples([('California', 2000), ('California', 2010),
                                   ('New York', 2000), ('New York', 2010),
                                   ('Texas', 2000), ('Texas', 2010)])
populations = [33871648, 37253956, 18976457, 19378102, 20851820, 25145561]
pop = pd.Series(populations, index=index)
pop.index.names = ['state', 'year']
print(pop)

# Clean partial indexing on the second level:
print(pop[:, 2010])

# MultiIndexed Series <-> DataFrame are the same info:
pop_df = pop.unstack()      # states as rows, years as columns
print(pop_df)
print(pop_df.stack())       # back to the MultiIndexed Series

In [ ]:
import numpy as np
import pandas as pd

# --- Several ways to build a MultiIndex ---
# Implicit: pass a list of index arrays
df = pd.DataFrame(np.random.rand(4, 2),
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
print(df)

print(pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]]))
print(pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)]))
print(pd.MultiIndex.from_product([['a', 'b'], [1, 2]]))  # Cartesian product

# A dict with tuple keys also auto-builds a MultiIndex:
data = {('California', 2000): 33871648, ('California', 2010): 37253956,
        ('Texas', 2000): 20851820,     ('Texas', 2010): 25145561}
print(pd.Series(data))

In [ ]:
import numpy as np
import pandas as pd

# --- MultiIndex on BOTH axes (acts like 4D data) ---
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# Some mock health data
np.random.seed(0)
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37
health_data = pd.DataFrame(data, index=index, columns=columns)
print(health_data)

# Partial column indexing by name:
print(health_data['Guido'])          # DataFrame for one subject
print(health_data['Guido', 'HR'])    # Series of one subject's heart rate

In [ ]:
import numpy as np
import pandas as pd

# --- IndexSlice for slicing across multiple levels ---
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])
np.random.seed(0)
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37
health_data = pd.DataFrame(data, index=index, columns=columns)

idx = pd.IndexSlice
print(health_data.loc[idx[:, 1], idx[:, 'HR']])  # visit 1, all HR columns

In [ ]:
import numpy as np
import pandas as pd

# --- Sorted indices are required for range slicing ---
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']

try:
    data['a':'b']             # fails: index is NOT lexicographically sorted
except Exception as e:
    print("Error:", type(e).__name__)

data = data.sort_index()      # fix it
print(data['a':'b'])          # now works

In [ ]:
import numpy as np
import pandas as pd

# --- reset_index / set_index: flat tables <-> multi-index ---
index = pd.MultiIndex.from_tuples([('California', 2000), ('California', 2010),
                                   ('New York', 2000), ('New York', 2010)],
                                  names=['state', 'year'])
pop = pd.Series([33871648, 37253956, 18976457, 19378102], index=index)

# Index levels -> columns (give the values a name)
pop_flat = pop.reset_index(name='population')
print(pop_flat)

# Columns -> index levels (the typical raw-data workflow)
print(pop_flat.set_index(['state', 'year']))

In [ ]:
import numpy as np
import pandas as pd

# --- Aggregating along a chosen level ---
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])
np.random.seed(0)
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37
health_data = pd.DataFrame(data, index=index, columns=columns)

# Book syntax uses the `level=` keyword on the reducer:
#     data_mean = health_data.mean(level='year')
#     data_mean.mean(axis=1, level='type')
# (added context) `level=` on reducers is removed in modern pandas (>=2.0).
# Use groupby instead, which is equivalent:
data_mean = health_data.groupby(level='year').mean()        # average over visits
print(data_mean)
print(data_mean.groupby(level='type', axis=1).mean())       # average HR / Temp

## Why this matters / intuition

Real-world data is frequently more than 2D (e.g. measurements indexed by *subject* x *type* over *year* x *visit*). Rather than reaching for awkward nested structures or external N-D arrays, hierarchical indexing keeps everything inside `Series`/`DataFrame`, so you retain all the alignment, slicing, and vectorized-operation machinery. The mental model: each extra index level is just another axis folded into a single labeled index, and `stack`/`unstack` let you "rotate" axes between the row index and the column index at will. This is also the natural output shape of `groupby` aggregations, so understanding `MultiIndex` is foundational for the chapters that follow.

## Gotchas

- **Sorting is mandatory for partial slices.** Range slicing (`data['a':'b']`) on an unsorted `MultiIndex` raises an `UnsortedIndexError` / `KeyError`. Call `sort_index()` first.
- **Don't put raw `slice(...)` / `:` inside an index tuple.** Python forbids `health_data.loc[(:, 1), (:, 'HR')]` syntactically. Use `pd.IndexSlice` (`idx[:, 1]`) instead.
- **Tuples-as-keys is a trap.** It looks fine for tiny data but is inefficient and lacks the optimized `MultiIndex` operations.
- **Partial indexing order matters.** `pop['California']` (top level) is easy; selecting a lower level (`pop[:, 2010]`) needs the leading colon.
- (added context) **`level=` on reducers (`.mean(level=...)`) is deprecated/removed** in pandas >= 2.0. Use `df.groupby(level=...).mean()`. The book predates this change.
- (added context) The book's low-level `pd.MultiIndex(levels=..., labels=...)` uses the old `labels=` argument; modern pandas renamed it to `codes=`.

## Suggested figure (optional)

A side-by-side diagram showing the *same* state/year population data in two forms: (left) a 1D `Series` with a two-level (`state`, `year`) `MultiIndex` drawn as indented row labels; (right) the result of `unstack()` — a 2D grid with `state` as row labels and `year` as column headers. Curved arrows labeled `unstack()` (left to right) and `stack()` (right to left) connect them, visually conveying that the two representations hold identical information and that stack/unstack just rotates a level between the row and column axes.

# Notes 15 — Combining Datasets: Concat and Append (Vanderplas Ch. 3.6)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.06-concat-and-append.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Combining datasets ranges from simple stacking to complex database-style joins. This section covers the simple end: `pd.concat`, which glues `Series` and `DataFrame` objects together along an axis. It builds on NumPy's `np.concatenate`, but adds index-aware behavior — Pandas tracks the row/column labels and lets you control what happens when they overlap or differ. (The richer database-style merges/joins are the next section's topic.)

## Key ideas / idioms

- **NumPy foundation:** `np.concatenate([arr1, arr2, ...], axis=0)` joins arrays along a chosen axis. Pandas generalizes this for labeled data.
- **`pd.concat` signature** (reference only — not runnable as written):
  ```
  pd.concat(objs, axis=0, join='outer', join_axes=None, ignore_index=False,
            keys=None, levels=None, names=None, verify_integrity=False, copy=True)
  ```
  - `objs` — list/tuple of objects to concatenate.
  - `axis` — `0` (default, stack rows) or `1` (stack columns).
  - `join` — `'outer'` (default, union of the other axis's labels) or `'inner'` (intersection).
  - `ignore_index` — `True` discards original indices and builds a fresh `0..n-1` integer index.
  - `keys` — labels each source so the result gets a hierarchical (MultiIndex) top level.
  - `verify_integrity` — `True` raises a `ValueError` if the result has duplicate indices.
- **Default preserves indices:** unlike `ignore_index`, the default keeps the original labels, so duplicates can result.
- **Three ways to deal with duplicate indices:**
  1. `verify_integrity=True` — catch them as an error.
  2. `ignore_index=True` — throw the old index away.
  3. `keys=[...]` — keep them but disambiguate via an outer MultiIndex level.
- **Joins on the off-axis labels:** when columns (or, for `axis=1`, rows) don't match, `'outer'` keeps the union (filling gaps with `NaN`) and `'inner'` keeps only shared labels.

There is no heavy math here; the operation is essentially a labeled set/sequence concatenation. Conceptually, for outer vs inner join on label sets $A$ and $B$:

$$\text{outer} = A \cup B, \qquad \text{inner} = A \cap B$$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# Helper used throughout the section to build small labeled DataFrames
def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

print(make_df('ABC', range(3)))

# --- NumPy foundation ---
x, y, z = [1, 2, 3], [4, 5, 6], [7, 8, 9]
print(np.concatenate([x, y, z]))          # [1 2 3 4 5 6 7 8 9]

a = [[1, 2], [3, 4]]
print(np.concatenate([a, a], axis=1))     # stacked side-by-side

# --- Series concatenation (indices preserved) ---
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
print(pd.concat([ser1, ser2]))

# --- DataFrame row-wise (axis=0, default) ---
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
print(pd.concat([df1, df2]))

# --- DataFrame column-wise (axis=1) ---
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])
print(pd.concat([df3, df4], axis=1))

In [ ]:
import numpy as np
import pandas as pd

def make_df(cols, ind):
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

# --- Duplicate indices ---
x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
y.index = x.index          # force overlapping indices

# 1) Catch duplicates
try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print("ValueError:", e)

# 2) Ignore the old index
print(pd.concat([x, y], ignore_index=True))   # fresh 0..3 index

# 3) Keep both via hierarchical keys
print(pd.concat([x, y], keys=['x', 'y']))     # MultiIndex top level x / y

In [ ]:
import numpy as np
import pandas as pd

def make_df(cols, ind):
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

# --- Joins when columns differ ---
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])

# Outer join (default): union of columns, missing entries -> NaN
print(pd.concat([df5, df6]))

# Inner join: intersection of columns (only B and C survive)
print(pd.concat([df5, df6], join='inner'))

# Reindex one frame to control the result columns
# (replacement for the old join_axes= argument; see Gotchas)
print(pd.concat([df5, df6.reindex(columns=df5.columns)]))

## Why this matters / intuition

Real analyses almost always pull data from multiple files, time periods, or sources, and the first step is to stack them coherently. `pd.concat` is the workhorse for "these tables describe the same things, just put them together" — append new months of records (axis=0) or attach new feature columns (axis=1). The index-awareness is the value-add over raw NumPy: it keeps labels aligned and gives you explicit control (`keys`, `ignore_index`, `join`) over the messy cases of overlapping or mismatched labels, so you don't silently misalign rows.

## Gotchas

- **Index duplication is silent by default.** `pd.concat` happily keeps repeated index labels; if you need uniqueness, reach for `ignore_index=True`, `keys=`, or `verify_integrity=True`.
- **`pd.concat` returns a copy** — it does not modify the inputs in place.
- **`axis='col'` shorthand:** the source uses `axis='col'`/`axis='index'` aliases, but `axis=1`/`axis=0` are the safest, most portable forms. *(added context)*
- **`df.append()` is deprecated.** The book shows `df1.append(df2)` as a convenience equivalent to `pd.concat([df1, df2])`. In modern pandas (deprecated in 1.4, removed in 2.0) `DataFrame.append`/`Series.append` no longer exist — use `pd.concat` instead. *(added context)*
- **Repeated appending is inefficient.** Each `concat`/append builds a whole new object, so growing a frame one piece at a time is $O(n^2)$-ish. Collect all the pieces in a list and call `pd.concat` once at the end.
- **`join_axes=` is gone.** The book's `pd.concat([df5, df6], join_axes=[df5.columns])` no longer works; achieve the same by reindexing a frame's off-axis labels before concatenating (as shown above). *(added context)*

## Suggested figure (optional)

A side-by-side diagram of two small tables and three result panels: (1) `axis=0` stacking them vertically into a taller table; (2) `axis=1` placing them side-by-side into a wider table; (3) an off-axis-label panel contrasting `join='outer'` (union of columns, NaN-filled gaps shaded) vs `join='inner'` (only the shared columns). Color-code each source so the reader can trace where every cell came from, and annotate the index column to highlight duplicate vs. `ignore_index` vs. `keys` (MultiIndex) outcomes.

# Notes 16 — Combining Datasets: Merge and Join (Vanderplas Ch. 3.7)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.07-merge-and-join.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Pandas implements high-performance in-memory joins through `pd.merge()`. The behavior is grounded in **relational algebra**, the formal set of rules for manipulating relational data that underlies most databases. `pd.merge()` combines two DataFrames by matching one or more **key** columns (or indices), aligning rows even when their original ordering differs.

## Key ideas / idioms

- **Three categories of joins**, defined by the multiplicity of the key in each table:
  - **One-to-one**: keys are unique in both tables → behaves like a column-wise concatenation with row alignment by key.
  - **Many-to-one**: the key is duplicated in one table → values from the "one" side are repeated to fill the "many" side.
  - **Many-to-many**: the key is duplicated in both tables → the result is the **Cartesian product** of matching rows per key.
- **Key specification**:
  - Default: `pd.merge(a, b)` auto-detects shared column name(s) as the key.
  - `on='col'` — explicit shared key column (must exist in both).
  - `left_on=`, `right_on=` — merge on differently named columns; leaves a redundant column to `drop`.
  - `left_index=True`, `right_index=True` — merge on the index; `df.join()` does this by default.
  - You can mix: e.g. `left_index=True, right_on='name'`.
- **`how=` controls set arithmetic** on the keys *(added context: think of the set of key values in each table)*:

$$
\text{inner} = L \cap R, \quad \text{outer} = L \cup R, \quad \text{left} = L, \quad \text{right} = R
$$

  - `inner` (default): only keys present in both; non-matches dropped.
  - `outer`: union of keys; missing fields filled with `NaN`.
  - `left` / `right`: keep all keys from the left / right table.
- **`suffixes=`**: when both tables share a non-key column name, pandas appends `_x` / `_y` by default; override with e.g. `suffixes=['_L', '_R']`.

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- One-to-one join (auto-detected key 'employee') ---
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
df3 = pd.merge(df1, df2)          # aligns on 'employee' despite different ordering
print(df3)

# --- Many-to-one join ('group' duplicated on the left after df3) ---
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
print(pd.merge(df3, df4))         # supervisor repeated to match each employee's group

# --- Many-to-many join ('group' duplicated in both) ---
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting', 'Engineering',
                              'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                               'spreadsheets', 'organization']})
print(pd.merge(df1, df5))         # each employee paired with all skills for their group

In [ ]:
import numpy as np
import pandas as pd

df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})

# --- on= : explicit key ---
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
print(pd.merge(df1, df2, on='employee'))

# --- left_on / right_on : differently named keys ---
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
merged = pd.merge(df1, df3, left_on='employee', right_on='name')
print(merged)
print(merged.drop('name', axis=1))   # drop the redundant duplicate column

# --- index-based merging ---
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
print(pd.merge(df1a, df2a, left_index=True, right_index=True))
print(df1a.join(df2a))               # join() merges on index by default

# --- mixing index and column ---
print(pd.merge(df1a, df3, left_index=True, right_on='name'))

In [ ]:
import numpy as np
import pandas as pd

# --- Set arithmetic via how= ---
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']})
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']})

print(pd.merge(df6, df7))                 # inner (default): only 'Mary'
print(pd.merge(df6, df7, how='outer'))    # union: NaN where missing
print(pd.merge(df6, df7, how='left'))     # all rows of df6
print(pd.merge(df6, df7, how='right'))    # all rows of df7

# --- suffixes for overlapping non-key columns ---
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'rank': [3, 1, 4, 2]})
print(pd.merge(df8, df9, on='name'))                          # rank_x, rank_y
print(pd.merge(df8, df9, on='name', suffixes=['_L', '_R']))  # rank_L, rank_R

## Why this matters / intuition

Real datasets rarely arrive in one tidy table; the common pattern is several sources keyed on a shared identifier (employee, state code, user id). Merge/join lets you assemble these into a single analysis-ready frame. Understanding the join as **set arithmetic over key values** makes the `how=` choice deliberate: use `inner` to keep only fully matched records, `outer` to retain everything and surface gaps as `NaN`. The book's capstone — combining state population, area, and abbreviation tables to compute 2010 population density — shows the realistic workflow: outer-merge with `left_on`/`right_on`, patch unmatched keys, drop helper columns, then a second merge for area before computing `population / area (sq. mi)`.

## Gotchas

- **Auto-detection merges on *all* shared column names** — if two tables happen to share an unintended column, the result is wrong. Be explicit with `on=` when in doubt.
- `left_on`/`right_on` leaves **two redundant columns** with identical data; `drop` one.
- Default join is **`inner`**, which silently discards non-matching rows — easy to lose data without noticing. Use `outer` if you need to detect mismatches.
- Many-to-many joins produce a **Cartesian product per key**, so row counts can explode unexpectedly.
- Overlapping non-key columns get `_x`/`_y` suffixes automatically; set `suffixes=` for readable names.
- *(added context: the book's population example uses `merged.drop('abbreviation', 1)` and `final.dropna(...)`; in current pandas the positional axis arg is deprecated — prefer `drop(columns='abbreviation')`.)*

## Suggested figure (optional)

A four-panel Venn-style diagram of two overlapping key sets L and R, one panel per `how=` value (inner = intersection shaded, outer = both circles shaded, left = L shaded, right = R shaded), with a tiny example table beneath each showing which rows survive and where `NaN` appears.

# Notes 17 — Aggregation and Grouping (Vanderplas Ch. 3.8)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Efficient summarization of large data means computing aggregations — `sum()`, `mean()`, `median()`, `min()`, `max()`, etc. Pandas Series/DataFrame objects expose these directly, returning one scalar per column by default. The real power comes from **`groupby`**, which conditions aggregations on subsets of the data via the **split-apply-combine** pattern. The chapter uses the Seaborn `planets` exoplanet dataset (1,035 rows, 6 columns: method, number, orbital_period, mass, distance, year) as its running example.

## Key ideas / idioms

**Simple aggregations.** For a Series, methods like `.sum()` and `.mean()` collapse to a single value. For a DataFrame, aggregates compute per column by default; pass `axis='columns'` to aggregate across columns within each row. `describe()` computes several aggregates at once and is great for first-pass exploration.

Common aggregation methods (the chapter's reference table):

| Method | Purpose |
|---|---|
| `count()` | Total number of items |
| `first()`, `last()` | First and last item |
| `mean()`, `median()` | Mean and median |
| `min()`, `max()` | Minimum and maximum |
| `std()`, `var()` | Standard deviation and variance |
| `mad()` | Mean absolute deviation |
| `prod()` | Product of all items |
| `sum()` | Sum of all items |

**Split-apply-combine.** The `groupby` operation has three conceptual steps:

$$\text{data} \;\xrightarrow{\text{split by key}}\; \{g_1, g_2, \dots\} \;\xrightarrow{\text{apply } f}\; \{f(g_1), f(g_2), \dots\} \;\xrightarrow{\text{combine}}\; \text{result}$$

- **Split:** break the table into groups based on the value of a key.
- **Apply:** run a function (aggregate / filter / transform / apply) within each group.
- **Combine:** merge the per-group results back into a single output indexed by the group keys.

`df.groupby('key')` returns a lazy `DataFrameGroupBy` object — no computation happens until you apply an aggregation. This avoids materializing intermediate per-group DataFrames *(added context: lazy evaluation also lets pandas dispatch the work efficiently in one pass)*.

**GroupBy object features.**
- *Column indexing:* `planets.groupby('method')['orbital_period']` selects a column from the grouped object, yielding a (still lazy) grouped Series.
- *Iteration:* iterating a GroupBy yields `(key, group)` pairs where each `group` is a sub-DataFrame.
- *Dispatch methods:* any method not defined on GroupBy is dispatched to each group (e.g. `.describe()`).

**Aggregate / filter / transform / apply** — the four core "apply" operations:
- `aggregate()` takes a string, function, list of these, or a `{column: func}` dict for column-specific aggregation.
- `filter()` keeps or drops entire groups based on a boolean-returning function of the group.
- `transform()` returns data the **same shape** as the input (e.g. center within group).
- `apply()` runs an arbitrary function on each group's DataFrame and combines the results.

**Specifying split keys.** The key can be a column name, a list/array/Series of group labels (length must match), a dict/Series mapping index values to groups, or even a Python function applied to the index. Keys can be combined in a list to form a multi-index grouping.

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- Simple aggregations ---
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
print(ser.sum(), ser.mean())

df = pd.DataFrame({'A': rng.rand(5), 'B': rng.rand(5)})
print(df.mean())                 # per-column (default axis)
print(df.mean(axis='columns'))   # per-row

In [ ]:
import numpy as np
import pandas as pd

# --- Split-apply-combine basics ---
df = pd.DataFrame({'key':  ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)})
print(df.groupby('key'))        # lazy DataFrameGroupBy object
print(df.groupby('key').sum())  # combine -> one row per key

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
df = pd.DataFrame({'key':   ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data1': range(6),
                   'data2': rng.randint(0, 10, 6)})

# Iteration over groups: each group is a sub-DataFrame
for (key, group) in df.groupby('key'):
    print("{0:>3s}  shape={1}".format(key, group.shape))

# Dispatch: describe() is applied per group
print(df.groupby('key')['data1'].describe())

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
df = pd.DataFrame({'key':   ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data1': range(6),
                   'data2': rng.randint(0, 10, 6)})

# aggregate: list of functions, and per-column dict
print(df.groupby('key').aggregate(['min', np.median, max]))
print(df.groupby('key').aggregate({'data1': 'min', 'data2': 'max'}))

# filter: keep groups whose data2 std exceeds 4
def filter_func(x):
    return x['data2'].std() > 4
print(df.groupby('key').filter(filter_func))

# transform: center each group (output same shape as input)
print(df.groupby('key').transform(lambda x: x - x.mean()))

# apply: normalize data1 by the group's data2 sum
def norm_by_data2(x):
    x['data1'] = x['data1'] / x['data2'].sum()
    return x
print(df.groupby('key').apply(norm_by_data2))

In [ ]:
import numpy as np
import pandas as pd

df2 = pd.DataFrame({'data1': range(6), 'data2': [5, 0, 3, 3, 7, 9]},
                   index=['A', 'B', 'C', 'A', 'B', 'C'])

# key as a list/array of labels (length matches rows)
L = [0, 1, 0, 1, 2, 0]
df0 = pd.DataFrame({'data1': range(6)})
print(df0.groupby(L).sum())

# key as a dict mapping index -> group
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
print(df2.groupby(mapping).sum())

# key as a Python function applied to the index
print(df2.groupby(str.lower).mean())

# combine keys for a multi-index grouping
print(df2.groupby([str.lower, mapping]).mean())

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns   # provides the planets dataset

planets = sns.load_dataset('planets')

# Column indexing + per-group median
print(planets.groupby('method')['orbital_period'].median())

# Practical example: discoveries by method and decade
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'
table = planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)
print(table)

## Why this matters / intuition
`groupby` is the workhorse of exploratory data analysis: almost any "per-category" question — average sales per region, error rate per model, counts per class — is a split-apply-combine. Thinking in those three steps lets you reason about *what shape* the output should be: `aggregate` collapses each group to a row, `transform` preserves shape (ideal for feature engineering like group-wise normalization), `filter` subsets whole groups, and `apply` is the escape hatch for anything else. The decade-vs-method table shows how groupby + `unstack` turns raw rows into a readable pivot revealing trends (Radial Velocity dominated early; Transit surged in the 2010s).

## Gotchas
- The GroupBy object is **lazy** — printing it shows an object, not data; you must apply an operation to trigger computation.
- A list/array key must have **the same length as the rows** being grouped; a dict/Series key maps **index values** (not positions) to groups.
- `transform` must return output the **same shape** as its input; `aggregate` reduces; mixing these up causes shape/index errors.
- `apply` receives each group as a **DataFrame** and can return a scalar, Series, or DataFrame — pandas infers how to combine, which can surprise you. *(added context: classic `apply(norm_by_data2)` mutates and returns the frame; prefer building a new column to avoid in-place side effects.)*
- `axis='columns'` aggregates across columns per row — easy to forget and get column-wise results instead.
- `mad()` referenced in the table was deprecated/removed in later pandas versions *(added context, not stated in the source)*.

## Suggested figure (optional)
A three-panel split-apply-combine diagram: on the left, a small input table with a `key` column (rows colored by key value A/B/C). A **"split"** arrow fans the rows out into three stacked mini-tables, one per key. An **"apply"** arrow over each mini-table shows a function (e.g. `sum`) reducing it to a single value. A **"combine"** arrow funnels those three results back into one compact output table indexed by A/B/C. Color-coding each key consistently across all three stages makes the data flow visually obvious.
---

# Notes 18 — Pivot Tables (Vanderplas Ch. 3.9)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.09-pivot-tables.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

A **pivot table** is a spreadsheet-style operation that takes column-wise data and produces a
two-dimensional (or higher) summary table. It is essentially a multidimensional version of
`GroupBy` aggregation: you split data along *two or more* keys, aggregate each cell, and lay the
result out as a grid (one key on the rows, another on the columns). `pivot_table` exists because
the equivalent multi-key `groupby` + `unstack` chains get verbose and hard to read fast.

The section motivates this with the Seaborn **Titanic** dataset (survival by sex/class/age/fare)
and the CDC **births** dataset (births by year/decade/gender/weekday).

## Key ideas / idioms

- **Pivot table = grouped-aggregate laid out in 2D.** The single-key version is just GroupBy:
  `titanic.groupby('sex')[['survived']].mean()`.
- **Two keys via GroupBy is awkward:**
  `titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()`.
- **Same thing, readable:**
  `titanic.pivot_table('survived', index='sex', columns='class')`.
- **First positional arg = the values column** to aggregate; `index` = row key; `columns` = column key.
- **Default aggregation is the mean** (`aggfunc='mean'`).
- **Multi-level pivots:** pass *lists* (or binned series) for `index`/`columns` to get hierarchical
  rows/columns.
- **Bin a continuous variable before pivoting** with `pd.cut` (fixed edges) or `pd.qcut` (quantiles),
  then use the binned series as a key.
- **Decade-rounding idiom:** `10 * (year // 10)` floors a year to its decade via integer division.

Robust spread estimate used in the births cleanup (a sigma derived from the IQR of a Gaussian):

$$\sigma \approx 0.74 \,\bigl(Q_{75} - Q_{25}\bigr)$$

Outliers are then filtered with a 5-sigma window around the median $\mu = Q_{50}$:

$$\mu - 5\sigma \;<\; \text{births} \;<\; \mu + 5\sigma$$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- Small inline stand-in for seaborn's titanic so this runs standalone ---
# (added context): the book loads `sns.load_dataset('titanic')` which needs
# internet + seaborn. Here is a tiny synthetic frame with the same columns used.
rng = np.random.default_rng(0)
n = 200
titanic = pd.DataFrame({
    'survived': rng.integers(0, 2, n),
    'sex':      rng.choice(['female', 'male'], n),
    'class':    rng.choice(['First', 'Second', 'Third'], n),
    'age':      rng.uniform(1, 75, n),
    'fare':     rng.uniform(5, 250, n),
})

# 1) GroupBy -> the manual, verbose path
print(titanic.groupby('sex')[['survived']].mean())
print(titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack())

# 2) Equivalent pivot_table (default aggfunc = mean)
print(titanic.pivot_table('survived', index='sex', columns='class'))

# 3) Multi-level pivot: bin age with pd.cut, use as an extra row key
age = pd.cut(titanic['age'], [0, 18, 80])
print(titanic.pivot_table('survived', ['sex', age], 'class'))

# 4) Four-dimensional: also bin fare with pd.qcut and put it on the columns
fare = pd.qcut(titanic['fare'], 2)
print(titanic.pivot_table('survived', ['sex', age], [fare, 'class']))

# 5) Per-column aggregation via a dict, and margins (grand totals)
print(titanic.pivot_table(index='sex', columns='class',
                          aggfunc={'survived': 'sum', 'fare': 'mean'}))
print(titanic.pivot_table('survived', index='sex', columns='class', margins=True))

In [ ]:
import numpy as np
import pandas as pd

# --- Births example (CDC data). The book reads a real CSV from GitHub. ---
# (added context): replaced the remote CSV with a tiny inline frame so the
# decade-rounding, robust sigma-clip, and datetime-index idioms run standalone.
births = pd.DataFrame({
    'year':   [1969, 1969, 1975, 1975, 1988, 1988, 1999, 1999],
    'month':  [1, 1, 6, 6, 12, 12, 3, 3],
    'day':    [1.0, 1.0, 15.0, 15.0, 25.0, 25.0, 9.0, 9.0],
    'gender': ['F', 'M', 'F', 'M', 'F', 'M', 'F', 'M'],
    'births': [4000, 4200, 4500, 4700, 4300, 4600, 1, 999999],  # last two are outliers
})

# Decade column via integer-division trick
births['decade'] = 10 * (births['year'] // 10)

# Pivot: total births per decade, split by gender
print(births.pivot_table('births', index='decade', columns='gender', aggfunc='sum'))

# Robust outlier removal using IQR-derived sigma
quartiles = np.percentile(births['births'], [25, 50, 75])
mu = quartiles[1]
sig = 0.74 * (quartiles[2] - quartiles[0])
births = births.query('(births > @mu - 5 * @sig) & (births < @mu + 5 * @sig)')

# Build a proper datetime index, then derive day-of-week
births['day'] = births['day'].astype(int)
births.index = pd.to_datetime(10000 * births.year +
                              100 * births.month +
                              births.day, format='%Y%m%d')
births['dayofweek'] = births.index.dayofweek
print(births[['gender', 'births', 'dayofweek']])

## Why this matters / intuition

Pivot tables are the fastest way to answer "how does outcome Y vary jointly across categories A
and B?" — exactly the exploratory question you ask constantly in data analysis. The Titanic example
makes the payoff concrete: a single readable call exposes that survival depended strongly on the
*interaction* of sex and class, something a 1D summary would hide. Because the API is so compact,
pivoting becomes a cheap, reach-for-it move during EDA rather than a multi-line ritual.

## Gotchas

- **Argument order:** the first positional arg is the *values* column, not the index. Be explicit
  with `index=`/`columns=` when in doubt.
- **`aggfunc` defaults to mean** — easy to forget when you actually wanted `'sum'` or a count.
- **Binned keys carry interval labels** (e.g. `(0, 18]`) from `pd.cut`/`pd.qcut`; `qcut` splits by
  *quantiles* (equal counts), `cut` by *value edges* — different bin contents.
- **`@`-references in `query`:** `@mu`/`@sig` pull from local Python variables; without `@` pandas
  looks for columns named `mu`/`sig`.
- **The 0.74 factor is an approximation** for the Gaussian IQR→sigma conversion (added context: the
  exact constant is $1/(2\,\Phi^{-1}(0.75)) \approx 0.7413$); it is *robust* to outliers, which is
  the whole point before sigma-clipping.
- **`margins=True`** adds an "All" row/column of totals; rename via `margins_name`.
- **`dropna=True`** (default) drops all-NaN entries; use `fill_value` to substitute a value into
  empty cells instead of leaving NaN.

## Suggested figure (optional)

A small-multiples line chart of mean daily births versus day-of-year, one faint line per decade
overlaid, would visualize the seasonal birth pattern the section builds toward. Pair it with a
simple bar chart of mean births by day-of-week to show the weekday-vs-weekend dip — both flow
directly from the datetime index derived in the second code block.

# Notes 19 — Vectorized String Operations (Vanderplas Ch. 3.10)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.10-working-with-strings.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Pandas adds **vectorized string operations** through the `.str` accessor on a Series (or Index). NumPy's vectorization speeds up numeric arrays element-wise but has no clean path for arrays of strings, and plain Python loops/comprehensions break on missing data. The `.str` accessor exposes Python's familiar string methods (plus regex helpers) in a vectorized, missing-value-aware form.

The motivating pain point: a list comprehension over data containing `None` raises `AttributeError` (illustrative — this block intentionally fails, so it is shown as display, not run):

```
data = ['peter', 'Paul', None, 'MARY', 'gUIDO']
[s.capitalize() for s in data]   # AttributeError: 'NoneType' has no attribute 'capitalize'
```

Pandas Series carry a `.str` attribute that applies the operation element-wise and **skips missing values gracefully**.

## Key ideas / idioms
- `.str` mirrors nearly all Python `str` methods, just vectorized: `lower`, `upper`, `capitalize`, `len`, `startswith`, `strip`, `split`, etc.
- Return type follows the operation: strings stay strings, `len()` gives integers, `is*` tests give booleans.
- A second family wraps Python's `re` module: `match`, `contains`, `extract`, `findall`, `count`, `replace`, `split`.
- A miscellaneous family handles indexing and reshaping: `get`, `slice`, indexing via `str[...]`, `get_dummies`, plus `cat`, `repeat`, `pad`, `wrap`, `join`, `normalize`.
- Chaining works: `s.str.split().str.get(-1)` re-applies `.str` after a method that returns lists.
- Boolean results from `contains`/`match` slot directly into masking and counting (`.sum()` counts `True`).

There is no real math in this section; the only "quantity" idiom is counting boolean hits:
$$\text{count} = \sum_i \mathbb{1}[\text{pattern matches } x_i].$$

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

# --- The .str accessor handles missing values ---
data = ['peter', 'Paul', None, 'MARY', 'gUIDO']
names = pd.Series(data)
print(names.str.capitalize())   # None is preserved, not an error

# --- Python-like string methods ---
monte = pd.Series(['Graham Chapman', 'John Cleese', 'Terry Gilliam',
                   'Eric Idle', 'Terry Jones', 'Michael Palin'])
print(monte.str.lower())
print(monte.str.len())                 # integers
print(monte.str.startswith('T'))       # booleans
print(monte.str.split())               # lists of words

# --- Regex methods ---
# extract: pull the first group from each entry (first names here)
print(monte.str.extract('([A-Za-z]+)', expand=False))
# findall: e.g., names that start and end with a consonant
print(monte.str.findall(r'^[^AEIOU].*[^aeiou]$'))
# contains: boolean search
print(monte.str.contains('Terry'))

# --- get / slice / indexing ---
print(monte.str[0:3])                  # first 3 chars of each
print(monte.str.split().str.get(-1))   # last name (last token)

# --- get_dummies: split coded indicators into columns ---
# (added context) inline data standing in for the recipe database,
# which the book downloads from a large JSON file.
full = pd.DataFrame({
    'name': monte,
    'info': ['B|C|D', 'B|D', 'A|C', 'B|C', 'B|C|D', 'B|D'],
})
print(full['info'].str.get_dummies('|'))

# --- Counting boolean hits (mirrors the recipe analysis) ---
desc = pd.Series(['quick breakfast bowl', 'hearty dinner', 'Breakfast tacos'])
print(desc.str.contains('[Bb]reakfast').sum())   # -> 2

## Why this matters / intuition
Real-world data is messy and text-heavy, and Vanderplas stresses that **cleaning/munging often makes up the majority of data-science work**. The `.str` accessor lets you express that cleaning concisely and safely: no manual loops, no crashes on `None`, and regex power on tap. The recipe-database example shows the payoff — boolean `str.contains` filters turn ~173,000 recipes into a tiny targeted set, which is the kernel of a simple ingredient-based recommender.

## Gotchas
- Methods that return lists (`split`, `findall`) require a second `.str` to keep operating element-wise (e.g., `.str.split().str.get(-1)`).
- `extract` needs a capture group `(...)`; `expand=False` returns a Series instead of a DataFrame.
- Regex methods are case-sensitive by default — the book uses `[Bb]reakfast`/`[Cc]innamon` character classes to catch capitalization; misspellings (e.g., "cinamon") are simply missed unless you account for them.
- Missing values propagate as `NaN`/`None` through `.str` operations rather than raising — convenient, but watch for them in downstream counts.
- `.str` is for object/string dtype; calling it on numeric data won't help.

## Suggested figure (optional)
A two-column "cheat-sheet" graphic: left column lists the Python-like methods grouped by return type (string-returning, integer-returning `len`, boolean `is*`/`startswith`); right column lists the regex family (`match`, `contains`, `extract`, `findall`, `count`, `replace`, `split`) with a one-word purpose beside each, plus a small footer box showing the `get_dummies('|')` split turning one coded column into several 0/1 indicator columns.
---

# Notes 20 — Working with Time Series (Vanderplas Ch. 3.11)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.11-working-with-time-series.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary

Time series work in Python sits on three layers, from most flexible to most efficient:

1. **Native `datetime` / `dateutil`** — built-in `datetime` module (convenient methods) plus the third-party `dateutil` for flexible string parsing. Easy to use, but *not vectorized*, so they are slow on large arrays.
2. **NumPy `datetime64`** — encodes a date as a 64-bit integer for compact, vectorized operations. Requires rigid input (e.g. `'2015-07-04'`) and forces a tradeoff between time *resolution* (years → attoseconds) and maximum time *span*. Nanosecond (`'ns'`) is the practical default for modern dates.
3. **Pandas time tools** — `Timestamp`, `Period`, `Timedelta` (and their index versions) combine the ergonomics of `datetime` with the efficiency of `datetime64`. These power time-indexed `Series`/`DataFrame` objects, which is what you actually use day-to-day.

Three scalar types and their matching index types:

| Concept | Scalar | Index | Built by |
|---|---|---|---|
| A specific moment | `Timestamp` | `DatetimeIndex` | `pd.to_datetime`, `pd.date_range` |
| A fixed-frequency interval | `Period` | `PeriodIndex` | `.to_period()`, `pd.period_range` |
| A duration | `Timedelta` | `TimedeltaIndex` | date subtraction, `pd.timedelta_range` |

## Key ideas / idioms

- **`pd.to_datetime`** parses a wide variety of formats (and lists of mixed formats) into a `DatetimeIndex`.
- A `Series`/`DataFrame` with a `DatetimeIndex` supports **intuitive date-based slicing**: pass date strings, ranges of date strings, or even a coarser unit like a year (`data['2015']`) to select all matching rows.
- **`Period` arithmetic**: subtracting `Timestamp`s yields a `Timedelta` / `TimedeltaIndex`.
- **`to_period(freq)`** converts a `DatetimeIndex` into a `PeriodIndex` (intervals instead of instants).
- **Regular sequences**: `pd.date_range` (timestamps), `pd.period_range` (periods), `pd.timedelta_range` (durations). Each takes either `start, end` or `start, periods=N`, plus an optional `freq`.

### Frequency-code table

| Code | Meaning | Code | Meaning |
|------|---------|------|---------|
| `D`  | Calendar day | `B`  | Business day |
| `W`  | Weekly | `M`  | Month end |
| `BM` | Business month end | `Q`  | Quarter end |
| `BQ` | Business quarter end | `A`  | Year end |
| `BA` | Business year end | `H`  | Hours |
| `BH` | Business hours | `T` / `min` | Minutes |
| `S`  | Seconds | `L` / `ms` | Milliseconds |
| `U` / `us` | Microseconds | `N`  | Nanoseconds |

Variants and modifiers:

- **Period-start variants**: add `S` → `MS` (month start), `QS` (quarter start), `AS` (year start). The non-`S` codes (`M`, `Q`, `A`) mark the *end* of the period.
- **Anchored offsets**: append a month/day → `Q-JAN` (quarters anchored to January), `A-DEC` (years ending in December), `W-MON` (weeks anchored on Monday).
- **Combinations**: prefix a number and chain codes → `"2H30T"` for 2 hours 30 minutes.

### resample vs. asfreq

The primary difference:

$$
\text{resample()} \;=\; \text{data \textbf{aggregation}}, \qquad
\text{asfreq()} \;=\; \text{data \textbf{selection}}
$$

- **`resample()`** computes a statistic over each new bin. Downsampling → aggregate (`.mean()`, `.sum()`); upsampling → typically NaNs unless filled.
- **`asfreq()`** picks the value *at* each new time point. Missing points can be filled via `method='ffill'` (forward) or `method='bfill'` (backward).
- Example contrast: `goog.resample('BA').mean()` gives the *average over each year*, while `goog.asfreq('BA')` gives the *value at year-end*.

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

# --- Parsing mixed formats into a DatetimeIndex ---
dates = pd.to_datetime([datetime(2015, 7, 3), '4th of July, 2015',
                        '2015-Jul-6', '07-07-2015', '20150708'])
print(dates)

# DatetimeIndex -> PeriodIndex (intervals); subtraction -> TimedeltaIndex
print(dates.to_period('D'))
print(dates - dates[0])

In [ ]:
import numpy as np
import pandas as pd

# --- Date-based indexing & slicing ---
index = pd.DatetimeIndex(['2014-07-04', '2014-08-04',
                          '2015-07-04', '2015-08-04'])
data = pd.Series([0, 1, 2, 3], index=index)

print(data['2014-07-04':'2015-07-04'])   # range slice by date strings
print(data['2015'])                       # all of 2015

In [ ]:
import numpy as np
import pandas as pd

# --- Regular sequences ---
print(pd.date_range('2015-07-03', '2015-07-10'))          # start + end (daily)
print(pd.date_range('2015-07-03', periods=8))             # start + count
print(pd.date_range('2015-07-03', periods=8, freq='H'))   # hourly

print(pd.period_range('2015-07', periods=8, freq='M'))    # monthly periods
print(pd.timedelta_range(0, periods=10, freq='H'))        # hourly durations
print(pd.timedelta_range(0, periods=9, freq='2H30T'))     # custom: 2h30m steps

In [ ]:
import numpy as np
import pandas as pd
from pandas.tseries.offsets import BDay

# --- Frequency offsets: business days ---
print(pd.date_range('2015-07-01', periods=5, freq=BDay()))

In [ ]:
import numpy as np
import pandas as pd

# --- resample vs asfreq vs rolling ---
# (added context) The book uses downloaded Google stock prices ("goog").
# We synthesize a daily series with pd.date_range so this runs standalone.
rng = pd.date_range('2010-01-01', '2014-12-31', freq='B')   # business days
rng_seed = np.random.RandomState(0)
goog = pd.Series(500 + np.cumsum(rng_seed.randn(len(rng))), index=rng)

# Aggregation vs selection at business-year-end:
print(goog.resample('BA').mean())   # average price per year (aggregation)
print(goog.asfreq('BA'))            # price at each year-end (selection)

# Upsampling with forward-fill:
print(goog.asfreq('D', method='ffill').head())

# Rolling window: centered 365-day moving average (smoothing)
rolling = goog.rolling(365, center=True)
print(rolling.mean().dropna().head())

## Why this matters / intuition

- Real-world data (finance, sensors, logs, web traffic) is overwhelmingly time-indexed; pandas turns "select July 2015" or "give me the yearly average" into one-liners instead of manual index math.
- The `Timestamp` / `Period` / `Timedelta` trio maps cleanly onto the three questions you ask of time: *when* (instant), *which interval* (period), and *how long* (duration).
- **resample vs asfreq is the conceptual crux**: changing frequency is ambiguous — do you want a *summary* of each new bin, or the *snapshot* at each new tick? Knowing which one you mean prevents silently wrong charts.
- **Rolling windows** are the entry point to smoothing and trend extraction — averaging out short-term noise to reveal long-term structure.

## Gotchas

- **`datetime`/`dateutil` don't vectorize** — fine for a handful of dates, a bottleneck for large arrays; prefer the pandas/`datetime64` path at scale.
- **`datetime64` resolution vs. span tradeoff**: the chosen unit caps the representable date range; `'ns'` is standard but has a narrower span than coarser units.
- **End vs. start codes**: `M`/`Q`/`A` land on the *end* of the period; you need `MS`/`QS`/`AS` for the *start*. Easy to be off by a full period.
- **`resample` upsampling produces NaN** at the newly introduced finer points unless you aggregate or fill — don't assume values appear automatically.
- **`asfreq` selects, it does not aggregate** — it returns only the exact values at the requested ticks (with optional `ffill`/`bfill`), which can drop data if your grid doesn't line up.
- **Centered rolling windows** leave NaNs at both ends (no full window available), so plots/stats should account for the trimmed edges.

## Suggested figure (optional)

A single line chart of the synthetic daily `goog` series (thin, light line) overlaid with its centered 365-day rolling mean (thick, dark line). Add markers at each `asfreq('BA')` year-end point and short horizontal segments showing each `resample('BA').mean()` yearly average. The visual contrast — wiggly raw data, smooth rolling trend, discrete year-end dots vs. flat yearly-average bars — makes the selection-vs-aggregation distinction immediately obvious.

# Notes 21 — High-Performance Pandas: eval() and query() (Vanderplas Ch. 3.12)

> **Source:** https://jakevdp.github.io/PythonDataScienceHandbook/03.12-performance-eval-and-query.html
> *Grad-student reading notes (Claude-generated run-through). Faithful to the source; anything beyond the source is flagged* (added context).

## Concept summary
Vectorized NumPy/Pandas operations are fast, but **compound** expressions force the
allocation of full-size **temporary arrays** for every subexpression. Pandas exposes
`pd.eval()`, `DataFrame.eval()`, and `DataFrame.query()`, which use the **Numexpr**
library to evaluate string expressions element-by-element *without* materializing those
intermediates. The headline win is **memory** (and, secondarily, speed and readability)
— especially when temporaries would exceed available RAM or blow the CPU cache.

## Key ideas / idioms
- **Why compound expressions allocate temporaries.** For an expression like
  `(x > 0.5) & (y < 0.5)`, Python/NumPy effectively does (conceptual — not runnable standalone):
  ```
  tmp1 = (x > 0.5)
  tmp2 = (y < 0.5)
  mask = tmp1 & tmp2
  ```
  Every subexpression produces its own full-size array. For a chain of $N$ operations
  over arrays of length $M$, the peak memory cost scales like
  $$\text{peak memory} \sim \mathcal{O}(N \cdot M),$$
  because intermediates pile up before the final result is formed.
- **Numexpr fix.** Numexpr evaluates the expression element-by-element, so it never needs
  to build the full intermediate arrays. This keeps the working set small enough to stay
  in **CPU L1/L2 cache** (a few MB) instead of spilling to main memory.
- **`pd.eval(expr_string)`** — top-level function; refer to objects by their Python
  variable names inside the string. Supports:
  - arithmetic: `-df1 * df2 / (df3 + df4) - df5`
  - comparisons, incl. **chained**: `df1 < df2 <= df3 != df4`
  - bitwise `&`/`|` and the literal words `and`/`or`: `(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)`
  - object attributes / indexing: `df2.T[0] + df3.iloc[1]`
  - **Not supported:** function calls, conditionals, loops.
- **`DataFrame.eval(expr)`** — refer to **columns by bare name** (`A` not `df.A`); more
  succinct than `pd.eval`. Supports **column assignment** (`D = (A + B) / C`), including
  modifying an existing column, via `inplace=True`.
- **`@local_var`** — inside `DataFrame.eval()`/`query()`, prefix a Python local variable
  with `@` to distinguish it from a column name. *Not available in `pd.eval()`.*
- **`DataFrame.query(expr)`** — cleaner syntax for **filtering** rows; replaces
  `df[(df.A < 0.5) & (df.B < 0.5)]` with `df.query('A < 0.5 and B < 0.5')`. Also
  supports `@local_var`.

## Worked code examples (runnable)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(42)

# --- pd.eval(): compute on whole DataFrames, no big temporaries ---
nrows, ncols = 1000, 100
df1, df2, df3, df4, df5 = (
    pd.DataFrame(rng.rand(nrows, ncols)) for _ in range(5)
)

# Same result, two ways:
direct = df1 + df2 + df3 + df4
viaeval = pd.eval('df1 + df2 + df3 + df4')
print("pd.eval matches direct sum:", np.allclose(direct, viaeval))

# Supported operation classes
arith   = pd.eval('-df1 * df2 / (df3 + df4) - df5')
compare = pd.eval('df1 < df2 <= df3 != df4')          # chained comparisons
bitwise = pd.eval('(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)')
words   = pd.eval('(df1 < 0.5) and (df2 < 0.5) or (df3 < df4)')  # and/or literals
print("bitwise == words:", np.allclose(bitwise, words))

# Object attributes / indices
attr = pd.eval('df2.T[0] + df3.iloc[1]')
print("attribute expr shape:", attr.shape)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
df = pd.DataFrame(rng.rand(1000, 3), columns=['A', 'B', 'C'])

# DataFrame.eval(): columns by bare name
r1 = pd.eval("(df.A + df.B) / (df.C - 1)")   # via pd.eval
r2 = df.eval('(A + B) / (C - 1)')            # via df.eval (more succinct)
print("eval forms match:", np.allclose(r1, r2))

# Column assignment (create then modify in place)
df.eval('D = (A + B) / C', inplace=True)
print("created column D:\n", df.head(2))
df.eval('D = (A - B) / C', inplace=True)     # overwrite existing D
print("modified column D:\n", df.head(2))

# @ references a Python local variable, not a column
column_mean = df.mean(axis=1)
r3 = df.eval('A + @column_mean')
print("first value of A + @column_mean:", r3.iloc[0])

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(1)
df = pd.DataFrame(rng.rand(1000, 3), columns=['A', 'B', 'C'])

# query(): row filtering with clean syntax
mask_way = df[(df.A < 0.5) & (df.B < 0.5)]
query_way = df.query('A < 0.5 and B < 0.5')
print("query matches masking:", mask_way.equals(query_way))

# query() with a local variable via @
Cmean = df['C'].mean()
sel = df.query('A < @Cmean and B < @Cmean')
print("rows selected:", len(sel))

# Inspect memory footprint of the array backing the DataFrame
print("nbytes:", df.values.nbytes)

## Why this matters / intuition
- **Memory is the most predictable payoff.** Masking like `df[(df.A < 0.5) & (df.B < 0.5)]`
  internally builds `tmp1`, `tmp2`, then a combined `tmp3` before indexing. With large
  frames each temporary is a full copy; if those copies exceed RAM, `eval()`/`query()`
  go from "nice" to "necessary."
- **Cache behavior.** Keeping the working set inside the CPU's L1/L2 cache (a few MB)
  avoids slow cache-miss penalties; full temporaries push data out to main memory.
- **Readability.** `df.query('A < 0.5 and B < 0.5')` reads more cleanly than the
  bracketed boolean-mask form.

## Gotchas
- **Speed is not guaranteed for small data.** For modestly sized arrays the traditional
  methods may actually be *faster*; the real wins are memory savings and readability.
  Decide based on whether temporaries strain memory, not reflexively.
- **`@` only works in `DataFrame.eval()`/`query()`**, never in top-level `pd.eval()`.
- **`pd.eval()` is limited:** no function calls, no conditionals, no loops.
- **Column assignment needs `inplace=True`** to modify the DataFrame itself (otherwise
  you get a new object). *(added context: in modern pandas `df.eval('D = ...')` without
  `inplace` returns a copy; assign it back, e.g. `df = df.eval(...)`.)*
- **Name collisions:** a bare name inside `eval`/`query` resolves to a *column*; use `@`
  to force the Python-variable meaning.
- *(added context: VanderPlas notes the rough rule of thumb that `eval`/`query` pay off
  when the temporary arrays are a significant fraction of available system memory
  (gigabytes); below that, prefer plain operations for clarity.)*

## Suggested figure (optional)
A side-by-side memory diagram: **Left** — "Standard NumPy/Pandas": three stacked bars
labeled `tmp1`, `tmp2`, `tmp3` each the full array size, summing to a tall peak-memory
column. **Right** — "Numexpr (eval/query)": a single thin sliver (one cache-line's worth
of elements processed at a time) feeding directly into the final result, illustrating
that no full intermediates are allocated.
---